In [ ]:
import pandas as pd

print(
    pd.read_csv(
        r"G:\My Drive\skripsi\proje\proje\data\interim\hasil 1,\6. dataset_final_bersih.csv",
        nrows=3,
    ).columns.tolist()
)

['id', 'tanggal_wib', 'username', 'followers', 'verified', 'likes', 'replies', 'text_clean_raw', 'text_clean_bert']


In [26]:
TEXT_COL = "text_clean_bert"
DATE_COL = "tanggal_wib"

In [28]:
scored_df = DataProcessor.process_and_score(
    input_path=INPUT_FILE,
    output_dir=OUTPUT_FOLDER,
    pipeline=pipeline,
    text_col="text_clean_bert",
    date_col="tanggal_wib",
)

daily_nst = DataProcessor.aggregate_daily_nst(
    df=scored_df,
    output_dir=OUTPUT_FOLDER,
    date_col="tanggal_wib",
)
daily_nst.head()

2026-07-16 18:17:44 - INFO - Pembersihan tanggal: 0 baris gagal parse dibuang.
2026-07-16 18:17:44 - INFO - Mulai skoring sentimen untuk 14704 dokumen...
2026-07-16 18:17:49 - INFO - Progres: 1600/14704 dokumen
2026-07-16 18:17:53 - INFO - Progres: 3200/14704 dokumen
2026-07-16 18:17:59 - INFO - Progres: 4800/14704 dokumen
2026-07-16 18:18:04 - INFO - Progres: 6400/14704 dokumen
2026-07-16 18:18:10 - INFO - Progres: 8000/14704 dokumen
2026-07-16 18:18:15 - INFO - Progres: 9600/14704 dokumen
2026-07-16 18:18:20 - INFO - Progres: 11200/14704 dokumen
2026-07-16 18:18:26 - INFO - Progres: 12800/14704 dokumen
2026-07-16 18:18:31 - INFO - Progres: 14400/14704 dokumen
2026-07-16 18:18:32 - INFO - Distribusi label: {'neu': 8630, 'neg': 3813, 'pos': 2261}
2026-07-16 18:18:33 - INFO - Tersimpan: dataset_scored.csv
2026-07-16 18:18:33 - INFO - Agregasi NSt harian...
2026-07-16 18:18:33 - INFO - Tersimpan: nst_harian.csv | 153 hari | NSt min=-0.5714, max=0.5000


,tanggal,pos,neu,neg,n_total,NSt
0,2025-06-01,4,7,8,19,-0.210526
1,2025-06-02,1,54,14,69,-0.188406
2,2025-06-03,6,38,8,52,-0.038462
3,2025-06-04,6,27,6,39,0.000000
4,2025-06-05,6,26,7,39,-0.025641


In [ ]:
"""
FASE 2B: Re-Labeling Komparatif & Validasi Manual
=================================================
Ketiga model sudah diverifikasi punya jalur safetensors (aman untuk
torch < 2.6) dan mapping labelnya sudah dicek dari config resmi:
  1. mdhugol         : fine-tune indobenchmark/indobert-base-p1 (SmSA)
  2. w11wo_roberta   : Indonesian RoBERTa (SmSA), akurasi tinggi, populer
  3. ayame_bertcahya : fine-tune cahya/bert-base-indonesian-1.5G (SmSA),
                       akurasi 93,7% pada benchmark SmSA
"""
import logging
import platform
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

# ------------------------------------------------------------------
# KONFIGURASI
# ------------------------------------------------------------------
# SESUAIKAN path ini dengan yang sudah terbukti jalan di notebook Anda
INPUT_FILE = Path(
    r"G:\My Drive\skripsi\proje\proje\data\interim\hasil 1,\6. dataset_final_bersih.csv"
)
OUTPUT_FOLDER = Path(".")
MODELS_ROOT = Path(
    "./models_sentimen"
)  # cache lokal per model (offline setelah unduhan pertama)

TEXT_COL = "text_clean_bert"
DATE_COL = "tanggal_wib"

TAHAP = "A"  # "A" -> "B" -> "C" -> "D"
MODEL_FINAL = ""  # isi SETELAH Tahap C, contoh: "w11wo_roberta"

N_GOLD_PER_KELAS = 60  # 60 x 3 kelas = 180 tweet untuk dilabel manual
SEED = 42
BATCH_SIZE = 32
MAX_LEN = 128
USE_FP16 = True

# Kandidat model. revision menunjuk varian safetensors resmi (SFconvertbot)
# untuk repo yang branch utamanya hanya berisi pytorch_model.bin.
CANDIDATES: Dict[str, Dict] = {
    "mdhugol": {
        "model_id": "mdhugol/indonesia-bert-sentiment-classification",
        "revision": "refs/pr/2",
        # config hanya berisi LABEL_0/1/2; mapping resmi dari model card
        "label_override": {0: "pos", 1: "neu", 2: "neg"},
    },
    "w11wo_roberta": {
        "model_id": "w11wo/indonesian-roberta-base-sentiment-classifier",
        "revision": None,  # safetensors sudah ada di branch utama
        "label_override": None,  # id2label: positive/neutral/negative
    },
    "ayame_bertcahya": {
        "model_id": "ayameRushia/bert-base-indonesian-1.5G-sentiment-analysis-smsa",
        "revision": "refs/pr/7",
        "label_override": None,  # id2label: Positive/Neutral/Negative
    },
}

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
logger = logging.getLogger(__name__)

VALID_CLS = {"pos", "neu", "neg"}
SANITY_SAMPLES = [
    "IHSG ditutup menguat tajam hari ini",
    "kepanikan pasar memicu aksi jual masif",
    "volume perdagangan berjalan normal",
]


# ------------------------------------------------------------------
# SCORER (satu kelas untuk semua kandidat)
# ------------------------------------------------------------------
class SentimentScorer:
    def __init__(
        self,
        alias: str,
        model_id: str,
        revision: Optional[str] = None,
        label_override: Optional[Dict[int, str]] = None,
        local_root: Path = MODELS_ROOT,
        use_fp16: bool = True,
        batch_size: int = BATCH_SIZE,
        max_len: int = MAX_LEN,
    ) -> None:
        self.alias = alias
        self.batch_size = batch_size
        self.max_len = max_len
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        local_dir = local_root / alias

        if (local_dir / "config.json").exists():
            logger.info(f"[{alias}] Memuat dari cache lokal (offline): {local_dir}")
            self.tokenizer = AutoTokenizer.from_pretrained(local_dir)
            self.model = AutoModelForSequenceClassification.from_pretrained(local_dir)
        else:
            logger.info(
                f"[{alias}] Mengunduh {model_id} (revisi: {revision or 'main'}), sekali saja..."
            )
            kw = {"revision": revision} if revision else {}
            self.tokenizer = AutoTokenizer.from_pretrained(model_id, **kw)
            self.model = AutoModelForSequenceClassification.from_pretrained(
                model_id, use_safetensors=True if revision else None, **kw
            )
            local_dir.mkdir(parents=True, exist_ok=True)
            self.tokenizer.save_pretrained(local_dir)
            self.model.save_pretrained(local_dir)
            logger.info(f"[{alias}] Cache lokal tersimpan: {local_dir}")

        self.model.to(self.device)
        self.model.eval()
        if self.device.type == "cuda" and use_fp16:
            self.model = self.model.half()

        self._map_labels(label_override)
        labels, conf = self.predict(SANITY_SAMPLES)
        logger.info(
            f"[{alias}] Sanity: {list(zip(SANITY_SAMPLES, labels, np.round(conf, 3)))}"
        )

    def _map_labels(self, override: Optional[Dict[int, str]]) -> None:
        def norm(raw: str) -> Optional[str]:
            s = str(raw).lower()
            if "pos" in s:
                return "pos"
            if "neg" in s:
                return "neg"
            if "neu" in s or "net" in s:
                return "neu"
            return None

        id2label = self.model.config.id2label
        mapped = {int(k): norm(v) for k, v in id2label.items()}
        if None in mapped.values():
            if override is None:
                raise ValueError(
                    f"[{self.alias}] Label tidak dikenali: {id2label}. Butuh label_override."
                )
            if set(override.keys()) != {int(k) for k in id2label} or not set(
                override.values()
            ).issubset(VALID_CLS):
                raise ValueError(
                    f"[{self.alias}] label_override tidak valid: {override}"
                )
            mapped = dict(override)
        self.id_to_cls = mapped
        logger.info(f"[{self.alias}] Mapping label: {self.id_to_cls}")

    @torch.inference_mode()
    def predict(self, texts: Sequence[str]) -> Tuple[List[str], List[float]]:
        """Mengembalikan (label, confidence). Confidence = probabilitas
        softmax kelas terpilih, dihitung dalam FP32."""
        labels: List[str] = []
        confs: List[float] = []
        total = len(texts)
        cur = self.batch_size
        i = 0
        n_batch = 0

        while i < total:
            batch = list(texts[i : i + cur])
            try:
                enc = self.tokenizer(
                    batch,
                    padding=True,
                    truncation=True,
                    max_length=self.max_len,
                    return_tensors="pt",
                ).to(self.device)
                logits = self.model(**enc).logits.float()
                probs = torch.softmax(logits, dim=-1)
                conf, pred = torch.max(probs, dim=-1)
                labels.extend(self.id_to_cls[int(p)] for p in pred.cpu().tolist())
                confs.extend(conf.cpu().tolist())
                i += len(batch)
                n_batch += 1
                if n_batch % 50 == 0:
                    logger.info(f"[{self.alias}] Progres: {i}/{total}")
            except RuntimeError as e:
                if "out of memory" in str(e).lower() and cur > 1:
                    cur = max(1, cur // 2)
                    logger.warning(
                        f"[{self.alias}] CUDA OOM, batch diturunkan ke {cur}."
                    )
                    torch.cuda.empty_cache()
                    continue
                raise
        return labels, confs

    def close(self) -> None:
        """Bebaskan VRAM sebelum memuat model berikutnya."""
        del self.model
        del self.tokenizer
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


# ------------------------------------------------------------------
# UTILITAS DATA
# ------------------------------------------------------------------
def muat_dataset() -> pd.DataFrame:
    header = pd.read_csv(INPUT_FILE, nrows=0).columns.tolist()
    for c in (TEXT_COL, DATE_COL):
        if c not in header:
            raise KeyError(f"Kolom '{c}' tidak ada. Kolom tersedia: {header}")
    usecols = [c for c in ("id", TEXT_COL, DATE_COL) if c in header]
    df = pd.read_csv(INPUT_FILE, usecols=usecols)
    df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors="coerce")
    df = df.dropna(subset=[DATE_COL]).reset_index(drop=True)
    # row_key: kunci join yang aman untuk Excel (id Twitter 19 digit bisa
    # rusak dibulatkan Excel, jadi jangan pernah join lewat kolom id).
    df["row_key"] = df.index.astype(int)
    if "id" in df.columns:
        df["id"] = df["id"].astype(str)
    logger.info(f"Dataset dimuat: {len(df)} baris.")
    return df


# ------------------------------------------------------------------
# TAHAP A: SKOR SEMUA MODEL
# ------------------------------------------------------------------
def tahap_a() -> None:
    df = muat_dataset()
    texts = df[TEXT_COL].fillna("").astype(str).tolist()

    for alias, cfg in CANDIDATES.items():
        scorer = SentimentScorer(
            alias=alias,
            model_id=cfg["model_id"],
            revision=cfg["revision"],
            label_override=cfg["label_override"],
            use_fp16=USE_FP16,
        )
        labels, confs = scorer.predict(texts)
        df[f"label_{alias}"] = labels
        df[f"conf_{alias}"] = np.round(confs, 4)
        scorer.close()
        logger.info(
            f"[{alias}] Distribusi: {df[f'label_{alias}'].value_counts().to_dict()}"
        )

    aliases = list(CANDIDATES.keys())
    for a in range(len(aliases)):
        for b in range(a + 1, len(aliases)):
            agree = (df[f"label_{aliases[a]}"] == df[f"label_{aliases[b]}"]).mean()
            logger.info(f"Kesepakatan {aliases[a]} vs {aliases[b]}: {agree:.1%}")

    out = OUTPUT_FOLDER / "dataset_multilabel.csv"
    df.to_csv(out, index=False, encoding="utf-8-sig")
    logger.info(f"Tersimpan: {out}")


# ------------------------------------------------------------------
# TAHAP B: SAMPEL VALIDASI MANUAL (BUTA)
# ------------------------------------------------------------------
def tahap_b() -> None:
    df = pd.read_csv(OUTPUT_FOLDER / "dataset_multilabel.csv")
    # Stratifikasi pakai label mdhugol agar pos/neg cukup terwakili,
    # tapi label TIDAK disertakan di file (pelabelan buta).
    rng_parts = []
    for lab, grp in df.groupby("label_mdhugol"):
        n = min(N_GOLD_PER_KELAS, len(grp))
        rng_parts.append(grp.sample(n, random_state=SEED))
    sample = pd.concat(rng_parts).sample(frac=1, random_state=SEED)  # acak urutan

    out_cols = sample[["row_key", TEXT_COL]].copy()
    out_cols["label_manual"] = ""
    out = OUTPUT_FOLDER / "sampel_validasi_manual.csv"
    out_cols.to_csv(out, index=False, encoding="utf-8-sig")

    logger.info(f"Tersimpan: {out} ({len(out_cols)} baris)")
    print(
        "\nINSTRUKSI PELABELAN MANUAL\n"
        "1. Buka sampel_validasi_manual.csv di Excel.\n"
        "2. Isi kolom label_manual untuk tiap baris dengan salah satu: pos / neu / neg.\n"
        "   pos = nada positif, dukungan, optimisme, apresiasi\n"
        "   neg = marah, kritik, kecewa, takut, pesimis, sarkasme negatif\n"
        "   neu = faktual, berita, pengumuman, pertanyaan, tidak jelas arahnya\n"
        "3. JANGAN mengubah kolom row_key. Simpan tetap sebagai CSV.\n"
        "4. Setelah selesai, jalankan TAHAP = 'C'.\n"
    )


# ------------------------------------------------------------------
# TAHAP C: EVALUASI MODEL VS LABEL MANUAL
# ------------------------------------------------------------------
def tahap_c() -> None:
    from sklearn.metrics import accuracy_score, classification_report, f1_score

    multi = pd.read_csv(OUTPUT_FOLDER / "dataset_multilabel.csv")
    gold = pd.read_csv(OUTPUT_FOLDER / "sampel_validasi_manual.csv")

    gold["label_manual"] = gold["label_manual"].astype(str).str.strip().str.lower()
    gold = gold[gold["label_manual"].isin(VALID_CLS)]
    if len(gold) < 30:
        raise ValueError(
            f"Baru {len(gold)} baris berlabel valid (pos/neu/neg). "
            "Lengkapi dulu pelabelan manualnya sebelum Tahap C."
        )
    logger.info(f"Gold sample terpakai: {len(gold)} baris.")

    merged = gold.merge(multi, on="row_key", how="inner", suffixes=("", "_m"))
    hasil = []
    for alias in CANDIDATES.keys():
        y_true = merged["label_manual"]
        y_pred = merged[f"label_{alias}"]
        acc = accuracy_score(y_true, y_pred)
        f1m = f1_score(y_true, y_pred, average="macro")
        hasil.append((alias, acc, f1m))
        print(f"\n===== {alias} =====")
        print(f"Akurasi: {acc:.3f} | Macro-F1: {f1m:.3f}")
        print(classification_report(y_true, y_pred, zero_division=0))

    hasil.sort(key=lambda x: x[2], reverse=True)
    print("\nPERINGKAT (berdasarkan Macro-F1):")
    for alias, acc, f1m in hasil:
        print(f"  {alias:18s} macro-F1={f1m:.3f}  akurasi={acc:.3f}")
    print(
        f"\nKandidat terbaik: '{hasil[0][0]}'. Isi MODEL_FINAL dengan alias itu, "
        "lalu jalankan TAHAP = 'D'."
    )


# ------------------------------------------------------------------
# TAHAP D: FINALISASI DATASET & NST
# ------------------------------------------------------------------
def tahap_d() -> None:
    if MODEL_FINAL not in CANDIDATES:
        raise ValueError(
            f"MODEL_FINAL='{MODEL_FINAL}' tidak valid. Pilih salah satu: {list(CANDIDATES)}. "
            "Jalankan Tahap C dulu sebelum memutuskan."
        )
    df = pd.read_csv(OUTPUT_FOLDER / "dataset_multilabel.csv")
    df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors="coerce")
    df = df.dropna(subset=[DATE_COL])
    df["label_sent"] = df[f"label_{MODEL_FINAL}"]

    scored = OUTPUT_FOLDER / "dataset_scored.csv"
    df.to_csv(scored, index=False, encoding="utf-8-sig")
    logger.info(f"Tersimpan: {scored} (model: {MODEL_FINAL})")

    daily = (
        df.groupby(df[DATE_COL].dt.date)["label_sent"]
        .value_counts()
        .unstack(fill_value=0)
        .reset_index()
    )
    daily = daily.rename(columns={DATE_COL: "tanggal"})
    daily.columns.name = None
    for col in ("pos", "neu", "neg"):
        if col not in daily.columns:
            daily[col] = 0
    daily["n_total"] = daily[["pos", "neu", "neg"]].sum(axis=1)
    daily["NSt"] = np.where(
        daily["n_total"] > 0,
        (daily["pos"] - daily["neg"]) / daily["n_total"],
        0.0,
    )
    daily = daily.sort_values("tanggal").reset_index(drop=True)
    daily = daily[["tanggal", "pos", "neu", "neg", "n_total", "NSt"]]

    nst = OUTPUT_FOLDER / "nst_harian.csv"
    daily.to_csv(nst, index=False, encoding="utf-8-sig")
    logger.info(
        f"Tersimpan: {nst} | {len(daily)} hari | model={MODEL_FINAL} | "
        f"NSt min={daily['NSt'].min():.4f}, max={daily['NSt'].max():.4f}"
    )


# ------------------------------------------------------------------
# ENTRY POINT
# ------------------------------------------------------------------
if __name__ == "__main__":
    try:
        if TAHAP == "A":
            tahap_a()
        elif TAHAP == "B":
            tahap_b()
        elif TAHAP == "C":
            tahap_c()
        elif TAHAP == "D":
            tahap_d()
        else:
            raise ValueError("TAHAP harus 'A', 'B', 'C', atau 'D'.")
    except Exception as e:
        logger.error(f"Berhenti: {e}", exc_info=True)
        raise

2026-07-16 18:46:04 - INFO - Dataset dimuat: 14704 baris.
2026-07-16 18:46:04 - INFO - [mdhugol] Mengunduh mdhugol/indonesia-bert-sentiment-classification (revisi: refs/pr/2), sekali saja...
2026-07-16 18:46:05 - INFO - HTTP Request: HEAD https://huggingface.co/mdhugol/indonesia-bert-sentiment-classification/resolve/refs%2Fpr%2F2/config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-16 18:46:05 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/mdhugol/indonesia-bert-sentiment-classification/62a4e339130eee4a68629da1edfd0abb894337e2/config.json "HTTP/1.1 200 OK"
2026-07-16 18:46:05 - INFO - HTTP Request: HEAD https://huggingface.co/mdhugol/indonesia-bert-sentiment-classification/resolve/refs%2Fpr%2F2/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-16 18:46:05 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/mdhugol/indonesia-bert-sentiment-classification/62a4e339130eee4a68629da1edfd0abb894337e2/tokenizer_config.json "HT

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

2026-07-16 18:46:10 - INFO - [mdhugol] Cache lokal tersimpan: models_sentimen\mdhugol
2026-07-16 18:46:11 - INFO - [mdhugol] Mapping label: {0: 'pos', 1: 'neu', 2: 'neg'}
2026-07-16 18:46:11 - INFO - [mdhugol] Sanity: [('IHSG ditutup menguat tajam hari ini', 'neu', np.float64(0.998)), ('kepanikan pasar memicu aksi jual masif', 'neu', np.float64(0.969)), ('volume perdagangan berjalan normal', 'neu', np.float64(0.699))]
2026-07-16 18:46:15 - INFO - [mdhugol] Progres: 1600/14704
2026-07-16 18:46:20 - INFO - [mdhugol] Progres: 3200/14704
2026-07-16 18:46:24 - INFO - [mdhugol] Progres: 4800/14704
2026-07-16 18:46:28 - INFO - [mdhugol] Progres: 6400/14704
2026-07-16 18:46:33 - INFO - [mdhugol] Progres: 8000/14704
2026-07-16 18:46:38 - INFO - [mdhugol] Progres: 9600/14704
2026-07-16 18:46:43 - INFO - [mdhugol] Progres: 11200/14704
2026-07-16 18:46:48 - INFO - [mdhugol] Progres: 12800/14704
2026-07-16 18:46:54 - INFO - [mdhugol] Progres: 14400/14704
2026-07-16 18:46:55 - INFO - [mdhugol] Distr

config.json:   0%|          | 0.00/929 [00:00<?, ?B/s]

...

2026-07-16 18:49:09 - INFO - HTTP Request: HEAD https://huggingface.co/ayameRushia/bert-base-indonesian-1.5G-sentiment-analysis-smsa/resolve/refs%2Fpr%2F7/chat_template.jinja "HTTP/1.1 404 Not Found"
2026-07-16 18:49:09 - INFO - HTTP Request: HEAD https://huggingface.co/ayameRushia/bert-base-indonesian-1.5G-sentiment-analysis-smsa/resolve/refs%2Fpr%2F7/config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-16 18:49:09 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/ayameRushia/bert-base-indonesian-1.5G-sentiment-analysis-smsa/620cba31b30b21b875e86890d0f2c8c52538e2d1/config.json "HTTP/1.1 200 OK"
2026-07-16 18:49:10 - INFO - HTTP Request: HEAD https://huggingface.co/ayameRushia/bert-base-indonesian-1.5G-sentiment-analysis-smsa/resolve/refs%2Fpr%2F7/model.safetensors "HTTP/1.1 302 Found"


model.safetensors:   0%|          | 0.00/443M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

2026-07-16 18:50:12 - INFO - [ayame_bertcahya] Cache lokal tersimpan: models_sentimen\ayame_bertcahya
2026-07-16 18:50:12 - INFO - [ayame_bertcahya] Mapping label: {0: 'pos', 1: 'neu', 2: 'neg'}
2026-07-16 18:50:12 - INFO - [ayame_bertcahya] Sanity: [('IHSG ditutup menguat tajam hari ini', 'neu', np.float64(1.0)), ('kepanikan pasar memicu aksi jual masif', 'neu', np.float64(1.0)), ('volume perdagangan berjalan normal', 'neu', np.float64(0.995))]
2026-07-16 18:50:15 - INFO - [ayame_bertcahya] Progres: 1600/14704
2026-07-16 18:50:19 - INFO - [ayame_bertcahya] Progres: 3200/14704
2026-07-16 18:50:22 - INFO - [ayame_bertcahya] Progres: 4800/14704
2026-07-16 18:50:26 - INFO - [ayame_bertcahya] Progres: 6400/14704
2026-07-16 18:50:29 - INFO - [ayame_bertcahya] Progres: 8000/14704
2026-07-16 18:50:33 - INFO - [ayame_bertcahya] Progres: 9600/14704
2026-07-16 18:50:36 - INFO - [ayame_bertcahya] Progres: 11200/14704
2026-07-16 18:50:40 - INFO - [ayame_bertcahya] Progres: 12800/14704
2026-07-16 1

In [ ]:
"""
FASE 2C: FILTER RELEVANSI KONTEKS (PASAR MODAL INDONESIA)
=========================================================
Dijalankan SEBELUM fase2e (leksikon & hybrid) dan SEBELUM penarikan
sampel validasi manual (gold sample).

Aturan keputusan (konservatif, presisi diutamakan):
  Tweet DIKELUARKAN hanya jika:
    (a) terdeteksi minimal satu kategori konteks asing, DAN
    (b) TIDAK memiliki jangkar pasar Indonesia (IHSG, BEI, rupiah,
        ticker emiten IDX, dst.).
  Tweet campuran ("Dow anjlok, IHSG ikut tertekan") DIPERTAHANKAN
  karena tetap memuat opini tentang pasar Indonesia.
  Kategori komoditas (emas, dolar) TIDAK dieksklusi, hanya dilaporkan:
  tweet "pindah ke emas karena IHSG anjlok" adalah sentimen pasar
  Indonesia yang sah.

Keluaran (non-destruktif, file masukan tidak diubah):
  1. dataset_multilabel_relevan.csv   -> masukan fase2e selanjutnya
  2. dataset_multilabel_eksklusi.csv  -> lampiran audit skripsi
  3. laporan_relevansi.txt            -> angka untuk Tabel 4.1 Bab IV

"""

import logging
import re
from pathlib import Path

import numpy as np
import pandas as pd

# ------------------------------------------------------------------
# KONFIGURASI
# ------------------------------------------------------------------
OUTPUT_FOLDER = Path(".")
FILE_INPUT = (
    OUTPUT_FOLDER
    / r"G:\My Drive\skripsi\proje\proje\data\interim\dataset_multilabel.csv"
)
FILE_RELEVAN = (
    OUTPUT_FOLDER
    / "G:\My Drive\skripsi\proje\proje\data\interim\dataset_multilabel_relevan.csv"
)
FILE_EKSKLUSI = (
    OUTPUT_FOLDER
    / "G:\My Drive\skripsi\proje\proje\data\interim\dataset_multilabel_eksklusi.csv"
)
FILE_LAPORAN = (
    OUTPUT_FOLDER / "G:\My Drive\skripsi\proje\proje\data\interim\laporan_relevansi.txt"
)

TEXT_COL = "text_clean_bert"
DATE_COL = "tanggal_wib"

# Jendela peristiwa menurut Bab III (untuk pemeriksaan sanitas tanggal).
# Sesuaikan bila jendela final di skripsi berubah.
EVENT_WINDOW = ("2025-03-01", "2025-09-30")

# Hasil putusan manual atas kategori lapor-saja (saham_metafora_pribadi,
# dll.): isi dengan kolom id tweet yang Anda putuskan keluar, contoh:
# EKSKLUSI_MANUAL_IDS = [1949012461727924531]
EKSKLUSI_MANUAL_IDS: list = []

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
logger = logging.getLogger(__name__)

# ------------------------------------------------------------------
# POLA DETEKSI
# ------------------------------------------------------------------
# (1) Konteks asing: pasar ekuitas luar negeri / indeks global
POLA_PASAR_LUAR = re.compile(
    r"saham (?:as|us|usa|amerika|amrik|jepang|china|tiongkok|korea|eropa|"
    r"australia|india|global|luar negeri|luar)\b"
    r"|(?:bursa|indeks?|index) (?:as|us|usa|amerika|asia|global|asing|eropa"
    r"|australia|wall street|saham (?:as|us|usa|amerika|australia|jepang|china|eropa|global))\b"
    r"|wall street|nasdaq|dow jones|s&p ?500|sp500|nikkei|hang seng"
    r"|kospi|nyse|ftse|dax\b|\basx\b|shanghai composite"
)
# (2) Konteks asing: aset kripto
POLA_KRIPTO = re.compile(
    r"\b(?:bitcoin|btc|ethereum|crypto|kripto|altcoin|binance|memecoin"
    r"|dogecoin|doge|solana|stablecoin|usdt)\b"
)
# (3) "bursa" bermakna non-pasar-modal (politik, sepak bola, lainnya)
POLA_BURSA_NONFIN = re.compile(
    r"bursa (?:transfer|caketum|capres|cawapres|cagub|cawagub|calon|ketum"
    r"|pelatih|pemain|kerja|jodoh|mobil|motor)\b"
)
# (3b) Bahasa Melayu / pasar Malaysia. Bab 3.4 sudah mengeksklusi
# "konten berbahasa selain Indonesia"; penanda dipilih yang TIDAK
# lazim dalam bahasa Indonesia (presisi diutamakan).
POLA_MELAYU = re.compile(
    r"\b(?:peratus|tempoh|pelaburan|syarikat|ringgit|klse|klci"
    r"|bursa malaysia|kwsp|pelabur|bilion|sahaja)\b"
    r"|\brm ?\d"
)
# (4) "saham" sebagai metafora reputasi pribadi (slang Twitter).
# "sahamnya" TIDAK dimasukkan pronomina karena lazim literal
# ("harga sahamnya turun"); varian "nya" hanya bila diikuti "di mata".
POLA_SAHAM_METAFORA = re.compile(
    r"saham ?(?:gue|gua|gw|aku|ku|lu|lo|kamu|dia|mu) (?:naik|turun|anjlok"
    r"|melesat|meroket|terjun)"
    r"|sahamnya (?:naik|turun|anjlok|meroket) di mata"
)
# (5) Hanya dilaporkan, TIDAK dieksklusi: komoditas / safe haven
POLA_KOMODITAS = re.compile(r"\b(?:emas|antam|logam mulia|xau)\b")

KATEGORI_EKSKLUSI = {
    "pasar_luar_negeri": POLA_PASAR_LUAR,
    "kripto": POLA_KRIPTO,
    "bursa_non_pasar_modal": POLA_BURSA_NONFIN,
    "bahasa_melayu_malaysia": POLA_MELAYU,
}
# Lapor saja, TIDAK dieksklusi otomatis: "saham gue turun 30%" bisa
# portofolio sungguhan. Periksa manual daftar di laporan.
KATEGORI_LAPOR = {
    "saham_metafora_pribadi": POLA_SAHAM_METAFORA,
    "komoditas": POLA_KOMODITAS,
}

# Jangkar pasar Indonesia: bila salah satu hadir, tweet DIPERTAHANKAN.
# Ticker dibatasi pada emiten besar/likuid yang tidak ambigu sebagai
# kata umum bahasa Indonesia (huruf kecil, sesuai text_clean_bert).
TICKER_IDX = (
    "bbca|bbri|bmri|bbni|bris|tlkm|asii|goto|brpt|adro|aadi|antm|mdka|"
    "ptba|pgas|smgr|unvr|icbp|indf|klbf|inco|tins|itmg|untr|cpin|excl|"
    "isat|towr|amrt|hmsp|ggrm|buka|emtk|arto|bbtn|pgeo|nckl|cuan\\b"
)
POLA_JANGKAR_ID = re.compile(
    r"\b(?:ihsg|jkse|jci|idx|bei\b|bursa efek|pasar modal|lq45|idx30"
    r"|rupiah|emiten|reksa ?dana|obligasi|sbn\b|ori\d{2}|sukuk"
    r"|" + TICKER_IDX + r")"
)


def deteksi(df: pd.DataFrame) -> pd.DataFrame:
    """Menambahkan kolom penanda kategori dan keputusan eksklusi."""
    t = df[TEXT_COL].astype(str).str.lower()
    for nama, pola in {**KATEGORI_EKSKLUSI, **KATEGORI_LAPOR}.items():
        df[f"flag_{nama}"] = t.str.contains(pola, na=False)
    df["punya_jangkar_id"] = t.str.contains(POLA_JANGKAR_ID, na=False)

    ada_asing = df[[f"flag_{k}" for k in KATEGORI_EKSKLUSI]].any(axis=1)
    df["eksklusi_manual"] = df["id"].isin(EKSKLUSI_MANUAL_IDS)
    df["eksklusi_konteks"] = (ada_asing & ~df["punya_jangkar_id"]) | df[
        "eksklusi_manual"
    ]
    return df


def cek_tanggal(df: pd.DataFrame, laporan: list) -> None:
    """Pemeriksaan sanitas rentang tanggal terhadap jendela peristiwa."""
    tgl = pd.to_datetime(df[DATE_COL], errors="coerce")
    lo, hi = tgl.min(), tgl.max()
    laporan.append(f"Rentang tanggal data : {lo.date()} s.d. {hi.date()}")
    laporan.append(f"Jendela Bab III      : {EVENT_WINDOW[0]} s.d. {EVENT_WINDOW[1]}")
    w0, w1 = pd.Timestamp(EVENT_WINDOW[0]), pd.Timestamp(EVENT_WINDOW[1])
    di_luar = ((tgl < w0) | (tgl > w1)).sum()
    if di_luar:
        laporan.append(
            f"PERINGATAN: {di_luar} tweet berada di luar jendela peristiwa "
            f"Bab III. Konfirmasi jendela final sebelum agregasi NSt; "
            f"file ini TIDAK memotong tanggal secara otomatis."
        )
    else:
        laporan.append("Seluruh tweet berada di dalam jendela peristiwa.")


def main() -> None:
    df = pd.read_csv(FILE_INPUT)
    if TEXT_COL not in df.columns:
        raise KeyError(f"Kolom '{TEXT_COL}' tidak ada di {FILE_INPUT}.")
    n0 = len(df)
    logger.info(f"Memuat {n0} baris dari {FILE_INPUT}")

    df = deteksi(df)
    laporan: list = []
    laporan.append("LAPORAN FILTER RELEVANSI KONTEKS (FASE 2F)")
    laporan.append("=" * 58)
    cek_tanggal(df, laporan)
    laporan.append("")
    laporan.append(f"Total tweet masuk                 : {n0}")

    laporan.append("")
    laporan.append("Deteksi per kategori (sebelum aturan jangkar):")
    for nama in KATEGORI_EKSKLUSI:
        n_flag = int(df[f"flag_{nama}"].sum())
        n_tertahan = int((df[f"flag_{nama}"] & df["punya_jangkar_id"]).sum())
        laporan.append(
            f"  {nama:26s}: {n_flag:5d} terdeteksi, "
            f"{n_tertahan:4d} dipertahankan (ada jangkar Indonesia)"
        )
    for nama in KATEGORI_LAPOR:
        laporan.append(
            f"  {nama + ' (lapor saja)':26s}: "
            f"{int(df[f'flag_{nama}'].sum()):5d} terdeteksi, tidak dieksklusi"
        )

    n_ekskl = int(df["eksklusi_konteks"].sum())
    laporan.append("")
    laporan.append(f"Tweet dikeluarkan (asing tanpa jangkar + manual) : {n_ekskl}")
    laporan.append(
        f"  di antaranya putusan manual (EKSKLUSI_MANUAL_IDS) : "
        f"{int(df['eksklusi_manual'].sum())}"
    )
    laporan.append(f"Tweet dipertahankan                     : {n0 - n_ekskl}")
    laporan.append("")
    laporan.append("Baris untuk Tabel 4.1 Bab IV:")
    laporan.append(
        f'  "Konten di luar konteks pasar modal Indonesia '
        f'(pasar luar negeri, kripto, makna non-finansial)" : {n_ekskl}'
    )

    # Contoh untuk audit manual / lampiran
    laporan.append("")
    laporan.append("CONTOH TWEET DIKELUARKAN (maks 8 per kategori):")
    for nama in KATEGORI_EKSKLUSI:
        sub = df[df["eksklusi_konteks"] & df[f"flag_{nama}"]]
        laporan.append(f"--- {nama} ({len(sub)} dikeluarkan) ---")
        for _, r in sub.head(8).iterrows():
            laporan.append(f"  - {str(r[TEXT_COL])[:110]}")

    laporan.append("")
    laporan.append(
        "DAFTAR LENGKAP saham_metafora_pribadi (putuskan manual, tweet DIPERTAHANKAN):"
    )
    for _, r in df[df["flag_saham_metafora_pribadi"]].iterrows():
        laporan.append(f"  - {str(r[TEXT_COL])[:110]}")

    laporan.append("")
    laporan.append("CONTOH CAMPURAN YANG DIPERTAHANKAN (jangkar Indonesia hadir):")
    campuran = df[
        df[[f"flag_{k}" for k in KATEGORI_EKSKLUSI]].any(axis=1)
        & df["punya_jangkar_id"]
    ]
    for _, r in campuran.head(8).iterrows():
        laporan.append(f"  - {str(r[TEXT_COL])[:110]}")

    # Simpan keluaran
    kolom_flag = [f"flag_{k}" for k in {**KATEGORI_EKSKLUSI, **KATEGORI_LAPOR}] + [
        "punya_jangkar_id",
        "eksklusi_konteks",
    ]
    df_ekskl = df[df["eksklusi_konteks"]].copy()
    df_rel = df[~df["eksklusi_konteks"]].drop(columns=kolom_flag)

    df_rel.to_csv(FILE_RELEVAN, index=False, encoding="utf-8-sig")
    df_ekskl.to_csv(FILE_EKSKLUSI, index=False, encoding="utf-8-sig")
    Path(FILE_LAPORAN).write_text("\n".join(laporan), encoding="utf-8")

    print("\n".join(laporan))
    logger.info(f"Tersimpan: {FILE_RELEVAN} ({len(df_rel)} baris)")
    logger.info(f"Tersimpan: {FILE_EKSKLUSI} ({len(df_ekskl)} baris)")
    logger.info(f"Tersimpan: {FILE_LAPORAN}")
    logger.info(
        "Langkah berikutnya: jalankan fase2e_leksikon_v4.py (TAHAP K) pada "
        "dataset_multilabel_relevan.csv, lalu tarik gold sample dari file itu."
    )


if __name__ == "__main__":
    main()

<>:48: SyntaxWarning: invalid escape sequence '\M'
<>:49: SyntaxWarning: invalid escape sequence '\M'
<>:50: SyntaxWarning: invalid escape sequence '\M'
<>:48: SyntaxWarning: invalid escape sequence '\M'
<>:49: SyntaxWarning: invalid escape sequence '\M'
<>:50: SyntaxWarning: invalid escape sequence '\M'
C:\Users\EZRA\AppData\Local\Temp\ipykernel_3236\2731002311.py:48: SyntaxWarning: invalid escape sequence '\M'
  FILE_RELEVAN = OUTPUT_FOLDER / "G:\My Drive\skripsi\proje\proje\data\interim\dataset_multilabel_relevan.csv"
C:\Users\EZRA\AppData\Local\Temp\ipykernel_3236\2731002311.py:49: SyntaxWarning: invalid escape sequence '\M'
  FILE_EKSKLUSI = OUTPUT_FOLDER / "G:\My Drive\skripsi\proje\proje\data\interim\dataset_multilabel_eksklusi.csv"
C:\Users\EZRA\AppData\Local\Temp\ipykernel_3236\2731002311.py:50: SyntaxWarning: invalid escape sequence '\M'
  FILE_LAPORAN = OUTPUT_FOLDER / "G:\My Drive\skripsi\proje\proje\data\interim\laporan_relevansi.txt"
2026-07-19 15:44:58 - INFO - Memuat 14

LAPORAN FILTER RELEVANSI KONTEKS (FASE 2F)
Rentang tanggal data : 2025-06-01 s.d. 2025-10-31
Jendela Bab III      : 2025-03-01 s.d. 2025-09-30
PERINGATAN: 1232 tweet berada di luar jendela peristiwa Bab III. Konfirmasi jendela final sebelum agregasi NSt; file ini TIDAK memotong tanggal secara otomatis.

Total tweet masuk                 : 14704

Deteksi per kategori (sebelum aturan jangkar):
  pasar_luar_negeri         :   321 terdeteksi,  191 dipertahankan (ada jangkar Indonesia)
  kripto                    :    33 terdeteksi,    7 dipertahankan (ada jangkar Indonesia)
  bursa_non_pasar_modal     :    69 terdeteksi,    6 dipertahankan (ada jangkar Indonesia)
  bahasa_melayu_malaysia    :   510 terdeteksi,   59 dipertahankan (ada jangkar Indonesia)
  saham_metafora_pribadi (lapor saja):     2 terdeteksi, tidak dieksklusi
  komoditas (lapor saja)    :    59 terdeteksi, tidak dieksklusi

Tweet dikeluarkan (asing tanpa jangkar + manual) : 654
  di antaranya putusan manual (EKSKLUSI_MANUAL

In [ ]:
"""
FASE 2D (REVISI): Leksikon V4 dan Perbandingan V1/V2/V3/V4
==========================================================
File ini MENGGANTIKAN fase2e_leksikon_v3.py. Dijalankan SETELAH
fase2f_relevansi_konteks.py (masukan default: dataset_multilabel_relevan.csv).

(semua bersifat TEKNIS, BUKAN kosakata baru;kosakata arah V3 tetap BEKU sesuai protokol):
  1. Disambiguasi subjek:
    - "bursa" tidak dihitung bila diikuti makna non-pasar-modal
       (transfer, caketum, capres, kerja, dst.).
    - "indeks" tidak dihitung bila diikuti makna non-pasar
       (prestasi, demokrasi, persepsi, massa tubuh, dst.).
    - "rupiah" tidak dihitung bila hanya nominal harga
       ("5 ribu rupiah", "16.500 rupiah"); konteks kurs tetap dihitung.
  2. Jendela negasi berhenti pada tanda baca kalimat, sehingga
     "kemarin gagal. ihsg menguat" tidak lagi ternegasi keliru.
  3. Pola persen minus tidak lagi salah tangkap rentang "naik 10-15%".
  4. Fitur emoji DINONAKTIFKAN: verifikasi empiris pada
     dataset_multilabel menunjukkan 0 kemunculan emoji chart di
     text_clean_bert (emoji terhapus saat pra-proses), jadi fitur ini
     kode mati yang menyesatkan pembaca metodologi.

"""

import logging
import re
from collections import Counter
from pathlib import Path
from typing import Dict, List, Optional

import numpy as np
import pandas as pd

# ------------------------------------------------------------------
# KONFIGURASI
# ------------------------------------------------------------------
OUTPUT_FOLDER = Path(".")
FILE_INPUT = (
    OUTPUT_FOLDER
    / r"G:\My Drive\skripsi\proje\proje\data\interim\dataset_multilabel_relevan.csv"
)  # keluaran fase2f
FILE_INPUT_FALLBACK = (
    OUTPUT_FOLDER
    / "G:\My Drive\skripsi\proje\proje\data\interim\dataset_multilabel .csv"
)  # darurat saja
FILE_LEKS = (
    OUTPUT_FOLDER
    / "G:\My Drive\skripsi\proje\proje\data\interim\dataset_multilabel_leks.csv"
)  # keluaran Tahap K
FILE_GOLD = (
    OUTPUT_FOLDER
    / "G:\My Drive\skripsi\proje\proje\data\interim\sampel_validasi_manual.csv"
)

TEXT_COL = "text_clean_bert"
DATE_COL = "tanggal_wib"
ALIAS_MODEL = ["mdhugol", "w11wo_roberta", "ayame_bertcahya"]

TAHAP = "K"  # "K" -> "E" -> "F"
MODEL_FINAL = ""  # isi SETELAH Tahap E, contoh: "hybrid4_mdhugol"

OVERRIDE_HANYA_NEU = True
NEGATOR = {"tidak", "tak", "bukan", "belum", "gagal", "tanpa", "batal"}

# ---------------- KOSAKATA (V1-V3 identik dengan fase2e lama; BEKU) ----------------
BEARISH_V1: List[str] = [
    "terjun bebas",
    "zona merah",
    "turun tajam",
    "koreksi dalam",
    "aksi jual",
    "jual masif",
    "tekanan jual",
    "cut loss",
    "dana asing keluar",
    "capital outflow",
    "ambrol",
    "anjlok",
    "ambruk",
    "jeblok",
    "rontok",
    "longsor",
    "merosot",
    "melemah",
    "memerah",
    "tumbang",
    "keok",
    "crash",
    "bearish",
    "koreksi",
    "terjun",
    "drop",
    "turun",
]
BULLISH_V1: List[str] = [
    "zona hijau",
    "naik tajam",
    "rekor tertinggi",
    "all time high",
    "cetak rekor",
    "aksi beli",
    "dana asing masuk",
    "capital inflow",
    "melesat",
    "melonjak",
    "meroket",
    "menguat",
    "rebound",
    "menghijau",
    "terbang",
    "bullish",
    "rally",
    "reli",
    "naik",
    "pulih",
    "bangkit",
]
BEARISH_V2: List[str] = BEARISH_V1 + [
    "terkoreksi",
    "menurun",
    "penurunan",
    "jatuh",
    "kejatuhan",
    "berjatuhan",
    "terjatuh",
    "pelemahan",
    "melorot",
    "amblas",
    "tertekan",
    "terpuruk",
    "terjungkal",
    "tergelincir",
    "tersungkur",
    "terperosok",
    "terhantam",
    "dihantam",
    "berguguran",
    "lesu",
    "loyo",
    "hancur",
    "negatif",
    "minus",
    "rugi",
    "merugi",
    "kerugian",
    "suram",
    "kabur",
    "nyungsep",
    "boncos",
    "berdarah",
    "arb",
    "suspensi",
    "disuspen",
    "trading halt",
    "panic selling",
    "sell off",
    "selloff",
    "bear market",
    "kepanikan",
    "panik",
    "resesi",
    "krisis",
]
BULLISH_V2: List[str] = BULLISH_V1 + [
    "penguatan",
    "kenaikan",
    "menanjak",
    "pemulihan",
    "terkerek",
    "terdongkrak",
    "melejit",
    "positif",
    "untung",
    "keuntungan",
    "profit",
    "hijau",
    "cuan",
    "ijo",
    "gacor",
    "moncer",
    "to the moon",
    "ara",
    "borong",
    "memborong",
    "diborong",
    "bull market",
]
BEARISH_V3: List[str] = BEARISH_V2 + [
    "net sell",
    "jual bersih",
    "outflow",
    "margin call",
    "cutloss",
    "auto reject bawah",
    "downtrend",
    "kolaps",
    "terkapar",
    "tergerus",
    "menyusut",
    "ambles",
    "ancur",
    "remuk",
    "jebol",
    "terendah",
    "bablas",
    "terdepresiasi",
    "nyusruk",
]
BULLISH_V3: List[str] = BULLISH_V2 + [
    "net buy",
    "beli bersih",
    "inflow",
    "auto reject atas",
    "uptrend",
    "ath",
    "serok",
    "buy the dip",
    "kokoh",
    "perkasa",
    "cerah",
    "terapresiasi",
    "menyala",
    "tertinggi",
]
# V4 memakai kosakata V3 tanpa perubahan (pembekuan dihormati).
BEARISH_V4, BULLISH_V4 = BEARISH_V3, BULLISH_V3

IDIOM_V1: List[str] = [
    "turun ke jalan",
    "turun tangan",
    "turun gunung",
    "naik daun",
    "naik pitam",
    "naik haji",
    "naik banding",
]
IDIOM_V2: List[str] = IDIOM_V1 + [
    "merah putih",
    "jatuh cinta",
    "jatuh tempo",
    "jatuh bangun",
    "korban jatuh",
    "krisis moral",
]
IDIOM_V3: List[str] = IDIOM_V2 + ["turun temurun", "naik kelas"]
IDIOM_V4: List[str] = IDIOM_V3

SUBJEK_V12: List[str] = ["ihsg", "saham", "bursa", "pasar", "indeks", "idx", "rupiah"]
SUBJEK_V3: List[str] = [
    "ihsg",
    "saham",
    "bursa",
    "pasar saham",
    "pasar modal",
    "pasar keuangan",
    "indeks",
    "idx",
    "rupiah",
    "porto",
    "portofolio",
    "emiten",
    "lq45",
]

# V4: subjek sebagai pola dengan disambiguasi konteks.
_BURSA_NONFIN = (
    "transfer|caketum|capres|cawapres|cagub|cawagub|calon|ketum|"
    "pelatih|pemain|kerja|jodoh|mobil|motor"
)
_INDEKS_NONPASAR = (
    "prestasi|massa|demokrasi|persepsi|kepuasan|pembangunan|"
    "glikemik|literasi|kebahagiaan"
)
SUBJEK_V4_POLA: List[str] = [
    r"\bihsg\b",
    r"\bsaham\b",
    r"\bpasar saham\b",
    r"\bpasar modal\b",
    r"\bpasar keuangan\b",
    r"\bidx\b",
    r"\bporto\b",
    r"\bportofolio\b",
    r"\bemiten\b",
    r"\blq45\b",
    rf"\bbursa\b(?!\s+(?:{_BURSA_NONFIN})\b)",
    rf"\bindeks\b(?!\s+(?:{_INDEKS_NONPASAR})\b)",
    # "rupiah" bukan nominal harga: tolak bila didahului digit/kata bilangan.
    r"(?<!\d\s)(?<!\d)(?<!ribu )(?<!juta )(?<!miliar )(?<!triliun )\brupiah\b",
]

EMOJI_BEAR = ["\U0001f4c9", "\U0001f53b"]
EMOJI_BULL = ["\U0001f4c8", "\U0001f680"]

POLA_MINUS = re.compile(r"(?:\bminus\s?|-)\d+(?:[.,]\d+)?\s?%")
POLA_PLUS = re.compile(r"\+\s?\d+(?:[.,]\d+)?\s?%")
# V4: tidak menangkap "-15%" pada rentang "10-15%".
POLA_MINUS_V4 = re.compile(r"(?<![\d-])(?:\bminus\s?|-)\d+(?:[.,]\d+)?\s?%")

VERSI: Dict[str, Dict] = {
    "v1": dict(
        bearish=BEARISH_V1,
        bullish=BULLISH_V1,
        idiom=IDIOM_V1,
        subjek=SUBJEK_V12,
        subjek_pola=None,
        numerik=False,
        flip_negasi=False,
        emoji=False,
        elongasi=False,
        negasi_kalimat=False,
        pola_minus=POLA_MINUS,
    ),
    "v2": dict(
        bearish=BEARISH_V2,
        bullish=BULLISH_V2,
        idiom=IDIOM_V2,
        subjek=SUBJEK_V12,
        subjek_pola=None,
        numerik=True,
        flip_negasi=False,
        emoji=False,
        elongasi=False,
        negasi_kalimat=False,
        pola_minus=POLA_MINUS,
    ),
    "v3": dict(
        bearish=BEARISH_V3,
        bullish=BULLISH_V3,
        idiom=IDIOM_V3,
        subjek=SUBJEK_V3,
        subjek_pola=None,
        numerik=True,
        flip_negasi=True,
        emoji=True,
        elongasi=True,
        negasi_kalimat=False,
        pola_minus=POLA_MINUS,
    ),
    "v4": dict(
        bearish=BEARISH_V4,
        bullish=BULLISH_V4,
        idiom=IDIOM_V4,
        subjek=None,
        subjek_pola=SUBJEK_V4_POLA,
        numerik=True,
        flip_negasi=True,
        emoji=False,
        elongasi=True,
        negasi_kalimat=True,
        pola_minus=POLA_MINUS_V4,
    ),
}
KOLOM_HYBRID = {"v1": "hybrid", "v2": "hybrid2", "v3": "hybrid3", "v4": "hybrid4"}

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
logger = logging.getLogger(__name__)
VALID_CLS = {"pos", "neu", "neg"}


# ------------------------------------------------------------------
# INTI LEKSIKON
# ------------------------------------------------------------------
def _token_sebelum(
    t: str, start: int, n: int = 2, stop_kalimat: bool = False
) -> List[str]:
    potong = t[:start]
    if stop_kalimat:
        potong = re.split(r"[.!?;,]", potong)[-1]
    return re.findall(r"[a-z0-9]+", potong)[-n:]


def _ada_subjek(t: str, cfg: Dict) -> bool:
    if cfg["subjek_pola"] is not None:
        return any(re.search(p, t) for p in cfg["subjek_pola"])
    return any(re.search(rf"\b{re.escape(s)}\b", t) for s in cfg["subjek"])


def label_leksikon(text: str, cfg: Dict) -> Optional[str]:
    t = str(text).lower()
    if cfg["elongasi"]:
        t = re.sub(r"(.)\1{2,}", r"\1", t)  # "turuuun" -> "turun"
    for idiom in cfg["idiom"]:
        t = t.replace(idiom, " ")

    if not _ada_subjek(t, cfg):
        return None

    def hitung(daftar: List[str]):
        hit, ternegasi = 0, 0
        for cue in daftar:
            for m in re.finditer(rf"\b{re.escape(cue)}\b", t):
                tok = _token_sebelum(t, m.start(), stop_kalimat=cfg["negasi_kalimat"])
                if set(tok) & NEGATOR:
                    ternegasi += 1
                else:
                    hit += 1
        return hit, ternegasi

    bear, _ = hitung(cfg["bearish"])  # bearish ternegasi dibuang
    bull, bull_neg = hitung(cfg["bullish"])
    if cfg["flip_negasi"]:
        bear += bull_neg  # "gagal menguat" = bearish

    if cfg["numerik"]:
        bear += len(cfg["pola_minus"].findall(t))
        bull += len(POLA_PLUS.findall(t))
    if cfg["emoji"]:
        bear += sum(t.count(e) for e in EMOJI_BEAR)
        bull += sum(t.count(e) for e in EMOJI_BULL)

    if bear > bull:
        return "neg"
    if bull > bear:
        return "pos"
    return None


# ------------------------------------------------------------------
# TAHAP K: HITUNG SEMUA VERSI, BANDINGKAN (NON-DESTRUKTIF)
# ------------------------------------------------------------------
def _muat_input() -> pd.DataFrame:
    if FILE_INPUT.exists():
        return pd.read_csv(FILE_INPUT)
    logger.warning(
        f"{FILE_INPUT} tidak ditemukan. Memakai {FILE_INPUT_FALLBACK}. "
        f"Jalankan fase2f_relevansi_konteks.py dulu agar konteks asing tersaring."
    )
    return pd.read_csv(FILE_INPUT_FALLBACK)


def tahap_k() -> None:
    df = _muat_input()
    wajib = [TEXT_COL] + [f"label_{a}" for a in ALIAS_MODEL]
    hilang = [c for c in wajib if c not in df.columns]
    if hilang:
        raise KeyError(f"Kolom {hilang} tidak ada. Jalankan Tahap A fase2b dulu.")

    for v in VERSI:
        logger.info(f"Menghitung leksikon {v}...")
        df[f"leks_{v}"] = df[TEXT_COL].map(lambda x, _v=v: label_leksikon(x, VERSI[_v]))

    print("\n=== CAKUPAN LEKSIKON ===")
    for v in VERSI:
        n = df[f"leks_{v}"].notna().sum()
        print(
            f"  {v}: {n:5d} tweet bersinyal | {df[f'leks_{v}'].value_counts().to_dict()}"
        )

    # Transparansi perubahan v3 -> v4 untuk lampiran skripsi
    hilang_v4 = df[df["leks_v3"].notna() & df["leks_v4"].isna()]
    baru_v4 = df[df["leks_v4"].notna() & df["leks_v3"].isna()]
    beda_arah = df[
        df["leks_v3"].notna() & df["leks_v4"].notna() & (df["leks_v3"] != df["leks_v4"])
    ]
    print(f"\n=== PERUBAHAN V3 -> V4 ===")
    print(f"  sinyal hilang (subjek palsu tersaring) : {len(hilang_v4)}")
    print(f"  sinyal baru                            : {len(baru_v4)}")
    print(f"  berubah arah                           : {len(beda_arah)}")
    for judul, sub in [
        ("HILANG DI V4", hilang_v4),
        ("BARU DI V4", baru_v4),
        ("BERUBAH ARAH", beda_arah),
    ]:
        if len(sub):
            print(f"  --- contoh {judul} (maks 8) ---")
            for _, r in sub.head(8).iterrows():
                print(
                    f"    [v3={r['leks_v3']} v4={r['leks_v4']}] {str(r[TEXT_COL])[:95]}"
                )

    for alias in ALIAS_MODEL:
        base = df[f"label_{alias}"]
        boleh = (
            base.eq("neu") if OVERRIDE_HANYA_NEU else pd.Series(True, index=df.index)
        )
        for v, pref in KOLOM_HYBRID.items():
            kol = f"label_{pref}_{alias}"
            df[kol] = np.where(df[f"leks_{v}"].notna() & boleh, df[f"leks_{v}"], base)
        u = {
            v: int((df[f"label_{p}_{alias}"] != base).sum())
            for v, p in KOLOM_HYBRID.items()
        }
        logger.info(
            f"[{alias}] koreksi v1={u['v1']} v2={u['v2']} v3={u['v3']} v4={u['v4']}"
        )

    sisa = df[df["label_hybrid4_mdhugol"].eq("neu")][TEXT_COL].astype(str).str.lower()
    sisa = sisa[sisa.str.contains("ihsg|saham|bursa", regex=True, na=False)]
    stop = set(
        "yang dan di ke dari ini itu dengan untuk pada juga ada tidak akan bisa "
        "sudah masih hari the of to in and ihsg saham bursa pasar indeks".split()
    )
    tokens = Counter(
        w for teks in sisa for w in re.findall(r"[a-z]{4,}", teks) if w not in stop
    )
    print(
        "\n=== 25 KATA TERSERING DI TWEET NEU TERSISA "
        "(satu-satunya sumber sah penambahan berikutnya) ==="
    )
    for w, c in tokens.most_common(25):
        print(f"  {w:20s} {c}")

    df.to_csv(FILE_LEKS, index=False, encoding="utf-8-sig")
    logger.info(f"Tersimpan: {FILE_LEKS} (file masukan tidak diubah)")


# ------------------------------------------------------------------
# TAHAP E: EVALUASI 15 PELABEL VS LABEL MANUAL
# ------------------------------------------------------------------
def tahap_e() -> None:
    from sklearn.metrics import accuracy_score, classification_report, f1_score

    multi = pd.read_csv(FILE_LEKS)
    gold = pd.read_csv(FILE_GOLD)
    gold["label_manual"] = gold["label_manual"].astype(str).str.strip().str.lower()
    gold = gold[gold["label_manual"].isin(VALID_CLS)]
    if len(gold) < 30:
        raise ValueError(
            f"Baru {len(gold)} baris gold valid. Lengkapi pelabelan manual dulu."
        )
    logger.info(f"Gold sample terpakai: {len(gold)} baris.")

    merged = gold.merge(multi, on="row_key", how="inner", suffixes=("", "_m"))
    if len(merged) < len(gold):
        logger.warning(
            f"{len(gold) - len(merged)} baris gold tidak ditemukan di dataset "
            f"relevan. Kemungkinan gold ditarik sebelum filter fase2f; "
            f"baris itu di luar populasi dan sah untuk gugur."
        )
    kandidat = [f"label_{a}" for a in ALIAS_MODEL]
    for pref in KOLOM_HYBRID.values():
        kandidat += [f"label_{pref}_{a}" for a in ALIAS_MODEL]
    kandidat = [k for k in kandidat if k in merged.columns]

    hasil = []
    for kol in kandidat:
        nama = kol.replace("label_", "", 1)
        y_true, y_pred = merged["label_manual"], merged[kol]
        acc = accuracy_score(y_true, y_pred)
        f1m = f1_score(y_true, y_pred, average="macro")
        hasil.append((nama, acc, f1m))
        print(f"\n===== {nama} =====")
        print(f"Akurasi: {acc:.3f} | Macro-F1: {f1m:.3f}")
        print(classification_report(y_true, y_pred, zero_division=0))

    hasil.sort(key=lambda x: x[2], reverse=True)
    print("\nPERINGKAT (Macro-F1):")
    for nama, acc, f1m in hasil:
        print(f"  {nama:26s} macro-F1={f1m:.3f}  akurasi={acc:.3f}")
    print(f"\nPemenang: '{hasil[0][0]}'. Isi MODEL_FINAL lalu jalankan TAHAP = 'F'.")


# ------------------------------------------------------------------
# TAHAP F: FINALISASI DATASET & NST
# ------------------------------------------------------------------
def tahap_f() -> None:
    pilihan = list(ALIAS_MODEL)
    for pref in KOLOM_HYBRID.values():
        pilihan += [f"{pref}_{a}" for a in ALIAS_MODEL]
    if MODEL_FINAL not in pilihan:
        raise ValueError(
            f"MODEL_FINAL='{MODEL_FINAL}' tidak valid. Pilihan: {pilihan}. "
            "Jalankan Tahap E dulu."
        )
    df = pd.read_csv(FILE_LEKS)
    df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors="coerce")
    df = df.dropna(subset=[DATE_COL])
    df["label_sent"] = df[f"label_{MODEL_FINAL}"]

    scored = OUTPUT_FOLDER / "dataset_scored.csv"
    df.to_csv(scored, index=False, encoding="utf-8-sig")
    logger.info(f"Tersimpan: {scored} (pelabel final: {MODEL_FINAL})")

    # Catatan konsistensi Bab III: rumus NSt di Bab III memakai
    # S_i = P(pos) - P(neg). Dengan label diskret (termasuk hasil hybrid),
    # S_i menjadi +1 (pos), 0 (neu), -1 (neg), sehingga
    # NSt = (n_pos - n_neg) / n_total adalah bentuk diskret rumus yang sama.
    # Jelaskan satu kalimat ini di Bab IV agar tidak dianggap ganti rumus.
    daily = (
        df.groupby(df[DATE_COL].dt.date)["label_sent"]
        .value_counts()
        .unstack(fill_value=0)
        .reset_index()
    )
    daily = daily.rename(columns={DATE_COL: "tanggal"})
    daily.columns.name = None
    for col in ("pos", "neu", "neg"):
        if col not in daily.columns:
            daily[col] = 0
    daily["n_total"] = daily[["pos", "neu", "neg"]].sum(axis=1)
    daily["NSt"] = np.where(
        daily["n_total"] > 0,
        (daily["pos"] - daily["neg"]) / daily["n_total"],
        0.0,
    )
    daily = daily.sort_values("tanggal").reset_index(drop=True)
    daily = daily[["tanggal", "pos", "neu", "neg", "n_total", "NSt"]]

    nst = OUTPUT_FOLDER / "nst_harian.csv"
    daily.to_csv(nst, index=False, encoding="utf-8-sig")
    logger.info(
        f"Tersimpan: {nst} | {len(daily)} hari | pelabel={MODEL_FINAL} | "
        f"NSt min={daily['NSt'].min():.4f}, max={daily['NSt'].max():.4f}"
    )


# ------------------------------------------------------------------
# ENTRY POINT
# ------------------------------------------------------------------
if __name__ == "__main__":
    try:
        if TAHAP == "K":
            tahap_k()
        elif TAHAP == "E":
            tahap_e()
        elif TAHAP == "F":
            tahap_f()
        else:
            raise ValueError("TAHAP harus 'K', 'E', atau 'F'.")
    except Exception as e:
        logger.error(f"Berhenti: {e}", exc_info=True)
        raise

<>:65: SyntaxWarning: invalid escape sequence '\M'
<>:66: SyntaxWarning: invalid escape sequence '\M'
<>:67: SyntaxWarning: invalid escape sequence '\M'
<>:65: SyntaxWarning: invalid escape sequence '\M'
<>:66: SyntaxWarning: invalid escape sequence '\M'
<>:67: SyntaxWarning: invalid escape sequence '\M'
C:\Users\EZRA\AppData\Local\Temp\ipykernel_3236\883459321.py:65: SyntaxWarning: invalid escape sequence '\M'
  FILE_INPUT_FALLBACK = OUTPUT_FOLDER / "G:\My Drive\skripsi\proje\proje\data\interim\dataset_multilabel .csv"  # darurat saja
C:\Users\EZRA\AppData\Local\Temp\ipykernel_3236\883459321.py:66: SyntaxWarning: invalid escape sequence '\M'
  FILE_LEKS = OUTPUT_FOLDER / "G:\My Drive\skripsi\proje\proje\data\interim\dataset_multilabel_leks.csv"       # keluaran Tahap K
C:\Users\EZRA\AppData\Local\Temp\ipykernel_3236\883459321.py:67: SyntaxWarning: invalid escape sequence '\M'
  FILE_GOLD = OUTPUT_FOLDER / "G:\My Drive\skripsi\proje\proje\data\interim\sampel_validasi_manual.csv"
2026-0


=== CAKUPAN LEKSIKON ===
  v1:  3454 tweet bersinyal | {'pos': 2052, 'neg': 1402}
  v2:  4696 tweet bersinyal | {'pos': 2779, 'neg': 1917}
  v3:  4871 tweet bersinyal | {'pos': 2915, 'neg': 1956}
  v4:  4856 tweet bersinyal | {'pos': 2928, 'neg': 1928}

=== PERUBAHAN V3 -> V4 ===
  sinyal hilang (subjek palsu tersaring) : 31
  sinyal baru                            : 16
  berubah arah                           : 2
  --- contoh HILANG DI V4 (maks 8) ---
    [v3=neg v4=nan] tipikal correction ihsg itu: 1) 10% kalau gak ada angin gak ada apa, ini kayak 2012, 2022, 2023
    [v3=neg v4=nan] 100jt itu kalau kalau cuma di fix income kaya reksadana itu sekitar 500rb ya jdi gini 100jt dib
    [v3=neg v4=nan] casa hari ini bergabung dengan breakout chart pattern yang sama seperti tbla kpig btps masih ad
    [v3=neg v4=nan] next nya apa saja nih? dari semua update diatas, tentu saja ada beberapa hal yang akan meluncur
    [v3=neg v4=nan] intinya saya masih pegang omongan akun ig ngerti saham. sa

2026-07-19 15:50:20 - INFO - Tersimpan: G:\My Drive\skripsi\proje\proje\data\interim\dataset_multilabel_leks.csv (file masukan tidak diubah)


In [ ]:
"""
FASE 2H: PIPELINE FINAL SATU FILE (MANDIRI, TANPA IMPOR ANTAR-FILE)
===================================================================
File ini MENGGANTIKAN fase2e_leksikon_v3/v4, fase2f, dan fase2g.
Tidak mengimpor file fase lain. Satu-satunya masukan wajib:
  dataset_multilabel.csv  (hasil fase2b Tahap A: kolom text_clean_bert,
                           tanggal_wib, row_key, label_mdhugol,
                           label_w11wo_roberta, label_ayame_bertcahya)

ISI PIPELINE :
  LANGKAH 1  Filter relevansi konteks: eksklusi tweet pasar luar negeri,
             kripto, "bursa" non-pasar-modal, dan bahasa Melayu/Malaysia,
             KECUALI ada jangkar pasar Indonesia (IHSG, BEI, rupiah,
             ticker IDX). Penegakan definisi populasi Bab 3.4.
  LANGKAH 2  Leksikon arah pasar V1-V5 (bearish -> neg, bullish -> pos).
             V4: perbaikan teknis (disambiguasi subjek, negasi berhenti
             di tanda baca, pola persen rentang, emoji nonaktif).
             V5: + kosakata hasil penambang korpus (nyangkut, amsiong,
             rungkad, delisting, suspend, disuspend | breakout,
             kinclong, multibagger). SETELAH V5 LEKSIKON BEKU.
  LANGKAH 3  Kolom hybrid per versi x 3 model IndoBERT (override hanya
             pada label "neu") + laporan bukti dan penambang kata.

PROTOKOL:
  TAHAP = "K" -> jalankan LANGKAH 1-3, tulis dataset_multilabel_final.csv
  TAHAP = "E" -> evaluasi 18 pelabel vs sampel_validasi_manual.csv
                 (gold sample ditarik dari dataset_multilabel_final.csv)
  TAHAP = "F" -> isi MODEL_FINAL dengan pemenang Tahap E ->
                 dataset_scored.csv + nst_harian.csv

Semua keluaran memakai nama baru, tidak menimpa file fase sebelumnya.
Dependensi: pandas, numpy, scikit-learn (tanpa torch)
"""

import logging
import re
from collections import Counter
from pathlib import Path
from typing import Dict, List, Optional

import numpy as np
import pandas as pd

# ==================================================================
# KONFIGURASI
# ==================================================================
OUTPUT_FOLDER = Path(".")
FILE_INPUT = (
    OUTPUT_FOLDER
    / r"G:\My Drive\skripsi\proje\proje\data\interim\dataset_multilabel_leks.csv"
)
FILE_FINAL = (
    OUTPUT_FOLDER
    / "G:\My Drive\skripsi\proje\proje\data\interim\dataset_multilabel_final.csv"
)
FILE_EKSKLUSI = (
    OUTPUT_FOLDER
    / "G:\My Drive\skripsi\proje\proje\data\interim\dataset_eksklusi_konteks.csv"
)
FILE_LAPORAN = (
    OUTPUT_FOLDER / "G:\My Drive\skripsi\proje\proje\data\interim\laporan_fase2h.txt"
)
FILE_GOLD = (
    OUTPUT_FOLDER
    / "G:\My Drive\skripsi\proje\proje\data\interim\sampel_validasi_manual.csv"
)

TEXT_COL = "text_clean_bert"
DATE_COL = "tanggal_wib"
ALIAS_MODEL = ["mdhugol", "w11wo_roberta", "ayame_bertcahya"]

TAHAP = "K"  # "K" -> "E" -> "F"
MODEL_FINAL = ""  # isi SETELAH Tahap E, contoh: "hybrid5_mdhugol"

# Versi yang dihitung. Untuk lari cepat cukup ["v4", "v5"];
# lengkap ["v1","v2","v3","v4","v5"] untuk tabel perbandingan skripsi.
HITUNG_VERSI: List[str] = ["v1", "v2", "v3", "v4", "v5"]

OVERRIDE_HANYA_NEU = True
NEGATOR = {"tidak", "tak", "bukan", "belum", "gagal", "tanpa", "batal"}

# Jendela peristiwa Bab III (pemeriksaan sanitas tanggal, tidak memotong)
EVENT_WINDOW = ("2025-03-01", "2025-09-30")

# Putusan manual: id tweet yang dikeluarkan berdasarkan audit manual
EKSKLUSI_MANUAL_IDS: List[int] = []

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
logger = logging.getLogger(__name__)
VALID_CLS = {"pos", "neu", "neg"}

# ==================================================================
# LANGKAH 1: POLA FILTER RELEVANSI KONTEKS
# ==================================================================
POLA_PASAR_LUAR = re.compile(
    r"saham (?:as|us|usa|amerika|amrik|jepang|china|tiongkok|korea|eropa|"
    r"australia|india|global|luar negeri|luar)\b"
    r"|(?:bursa|indeks?|index) (?:as|us|usa|amerika|asia|global|asing|eropa"
    r"|australia|wall street)\b"
    r"|wall street|nasdaq|dow jones|s&p ?500|sp500|nikkei|hang seng"
    r"|kospi|nyse|ftse|dax\b|\basx\b|shanghai composite"
)
POLA_KRIPTO = re.compile(
    r"\b(?:bitcoin|btc|ethereum|crypto|kripto|altcoin|binance|memecoin"
    r"|dogecoin|doge|solana|stablecoin|usdt)\b"
)
POLA_BURSA_NONFIN = re.compile(
    r"bursa (?:transfer|caketum|capres|cawapres|cagub|cawagub|calon|ketum"
    r"|pelatih|pemain|kerja|jodoh|mobil|motor)\b"
)
POLA_MELAYU = re.compile(
    r"\b(?:peratus|tempoh|pelaburan|syarikat|ringgit|klse|klci"
    r"|bursa malaysia|kwsp|pelabur|bilion|sahaja)\b"
    r"|\brm ?\d"
)
KATEGORI_EKSKLUSI = {
    "pasar_luar_negeri": POLA_PASAR_LUAR,
    "kripto": POLA_KRIPTO,
    "bursa_non_pasar_modal": POLA_BURSA_NONFIN,
    "bahasa_melayu_malaysia": POLA_MELAYU,
}
# Lapor saja (ambigu, putuskan manual lewat EKSKLUSI_MANUAL_IDS):
POLA_SAHAM_METAFORA = re.compile(
    r"saham ?(?:gue|gua|gw|aku|ku|lu|lo|kamu|dia|mu) (?:naik|turun|anjlok"
    r"|melesat|meroket|terjun)"
    r"|sahamnya (?:naik|turun|anjlok|meroket) di mata"
)
POLA_KOMODITAS = re.compile(r"\b(?:emas|antam|logam mulia|xau)\b")
KATEGORI_LAPOR = {
    "saham_metafora_pribadi": POLA_SAHAM_METAFORA,
    "komoditas": POLA_KOMODITAS,
}

TICKER_IDX = (
    "bbca|bbri|bmri|bbni|bris|tlkm|asii|goto|brpt|adro|aadi|antm|mdka|"
    "ptba|pgas|smgr|unvr|icbp|indf|klbf|inco|tins|itmg|untr|cpin|excl|"
    "isat|towr|amrt|hmsp|ggrm|buka|emtk|arto|bbtn|pgeo|nckl|cuan\\b"
)
POLA_JANGKAR_ID = re.compile(
    r"\b(?:ihsg|jkse|jci|idx|bei\b|bursa efek|pasar modal|lq45|idx30"
    r"|rupiah|emiten|reksa ?dana|obligasi|sbn\b|ori\d{2}|sukuk"
    r"|" + TICKER_IDX + r")"
)

# ==================================================================
# LANGKAH 2: KOSAKATA LEKSIKON (V1-V3 warisan; V5 tambahan; BEKU)
# ==================================================================
BEARISH_V1: List[str] = [
    "terjun bebas",
    "zona merah",
    "turun tajam",
    "koreksi dalam",
    "aksi jual",
    "jual masif",
    "tekanan jual",
    "cut loss",
    "dana asing keluar",
    "capital outflow",
    "ambrol",
    "anjlok",
    "ambruk",
    "jeblok",
    "rontok",
    "longsor",
    "merosot",
    "melemah",
    "memerah",
    "tumbang",
    "keok",
    "crash",
    "bearish",
    "koreksi",
    "terjun",
    "drop",
    "turun",
]
BULLISH_V1: List[str] = [
    "zona hijau",
    "naik tajam",
    "rekor tertinggi",
    "all time high",
    "cetak rekor",
    "aksi beli",
    "dana asing masuk",
    "capital inflow",
    "melesat",
    "melonjak",
    "meroket",
    "menguat",
    "rebound",
    "menghijau",
    "terbang",
    "bullish",
    "rally",
    "reli",
    "naik",
    "pulih",
    "bangkit",
]
BEARISH_V2: List[str] = BEARISH_V1 + [
    "terkoreksi",
    "menurun",
    "penurunan",
    "jatuh",
    "kejatuhan",
    "berjatuhan",
    "terjatuh",
    "pelemahan",
    "melorot",
    "amblas",
    "tertekan",
    "terpuruk",
    "terjungkal",
    "tergelincir",
    "tersungkur",
    "terperosok",
    "terhantam",
    "dihantam",
    "berguguran",
    "lesu",
    "loyo",
    "hancur",
    "negatif",
    "minus",
    "rugi",
    "merugi",
    "kerugian",
    "suram",
    "kabur",
    "nyungsep",
    "boncos",
    "berdarah",
    "arb",
    "suspensi",
    "disuspen",
    "trading halt",
    "panic selling",
    "sell off",
    "selloff",
    "bear market",
    "kepanikan",
    "panik",
    "resesi",
    "krisis",
]
BULLISH_V2: List[str] = BULLISH_V1 + [
    "penguatan",
    "kenaikan",
    "menanjak",
    "pemulihan",
    "terkerek",
    "terdongkrak",
    "melejit",
    "positif",
    "untung",
    "keuntungan",
    "profit",
    "hijau",
    "cuan",
    "ijo",
    "gacor",
    "moncer",
    "to the moon",
    "ara",
    "borong",
    "memborong",
    "diborong",
    "bull market",
]
BEARISH_V3: List[str] = BEARISH_V2 + [
    "net sell",
    "jual bersih",
    "outflow",
    "margin call",
    "cutloss",
    "auto reject bawah",
    "downtrend",
    "kolaps",
    "terkapar",
    "tergerus",
    "menyusut",
    "ambles",
    "ancur",
    "remuk",
    "jebol",
    "terendah",
    "bablas",
    "terdepresiasi",
    "nyusruk",
]
BULLISH_V3: List[str] = BULLISH_V2 + [
    "net buy",
    "beli bersih",
    "inflow",
    "auto reject atas",
    "uptrend",
    "ath",
    "serok",
    "buy the dip",
    "kokoh",
    "perkasa",
    "cerah",
    "terapresiasi",
    "menyala",
    "tertinggi",
]
# V5: hasil penambang korpus, diaudit dua arah. Ditolak dengan alasan:
# tersangkut (kasus hukum), gocap (arah campuran), ngamuk/bantai
# (tabrakan kosakata demo), dump (edukasi pump-and-dump), lainnya tipis.
TAMBAHAN_BEARISH_V5: List[str] = [
    "nyangkut",
    "amsiong",
    "rungkad",
    "delisting",
    "suspend",
    "disuspend",
]
TAMBAHAN_BULLISH_V5: List[str] = [
    "breakout",
    "kinclong",
    "multibagger",
]
BEARISH_V5: List[str] = BEARISH_V3 + TAMBAHAN_BEARISH_V5
BULLISH_V5: List[str] = BULLISH_V3 + TAMBAHAN_BULLISH_V5

IDIOM_V1: List[str] = [
    "turun ke jalan",
    "turun tangan",
    "turun gunung",
    "naik daun",
    "naik pitam",
    "naik haji",
    "naik banding",
]
IDIOM_V2: List[str] = IDIOM_V1 + [
    "merah putih",
    "jatuh cinta",
    "jatuh tempo",
    "jatuh bangun",
    "korban jatuh",
    "krisis moral",
]
IDIOM_V3: List[str] = IDIOM_V2 + ["turun temurun", "naik kelas"]

SUBJEK_V12: List[str] = ["ihsg", "saham", "bursa", "pasar", "indeks", "idx", "rupiah"]
SUBJEK_V3: List[str] = [
    "ihsg",
    "saham",
    "bursa",
    "pasar saham",
    "pasar modal",
    "pasar keuangan",
    "indeks",
    "idx",
    "rupiah",
    "porto",
    "portofolio",
    "emiten",
    "lq45",
]
_BURSA_NONFIN = (
    "transfer|caketum|capres|cawapres|cagub|cawagub|calon|ketum|"
    "pelatih|pemain|kerja|jodoh|mobil|motor"
)
_INDEKS_NONPASAR = (
    "prestasi|massa|demokrasi|persepsi|kepuasan|pembangunan|"
    "glikemik|literasi|kebahagiaan"
)
SUBJEK_V45_POLA: List[str] = [
    r"\bihsg\b",
    r"\bsaham\b",
    r"\bpasar saham\b",
    r"\bpasar modal\b",
    r"\bpasar keuangan\b",
    r"\bidx\b",
    r"\bporto\b",
    r"\bportofolio\b",
    r"\bemiten\b",
    r"\blq45\b",
    rf"\bbursa\b(?!\s+(?:{_BURSA_NONFIN})\b)",
    rf"\bindeks\b(?!\s+(?:{_INDEKS_NONPASAR})\b)",
    r"(?<!\d\s)(?<!\d)(?<!ribu )(?<!juta )(?<!miliar )(?<!triliun )\brupiah\b",
]

POLA_MINUS = re.compile(r"(?:\bminus\s?|-)\d+(?:[.,]\d+)?\s?%")
POLA_MINUS_V45 = re.compile(r"(?<![\d-])(?:\bminus\s?|-)\d+(?:[.,]\d+)?\s?%")
POLA_PLUS = re.compile(r"\+\s?\d+(?:[.,]\d+)?\s?%")

VERSI: Dict[str, Dict] = {
    "v1": dict(
        bearish=BEARISH_V1,
        bullish=BULLISH_V1,
        idiom=IDIOM_V1,
        subjek=SUBJEK_V12,
        subjek_pola=None,
        numerik=False,
        flip_negasi=False,
        elongasi=False,
        negasi_kalimat=False,
        pola_minus=POLA_MINUS,
    ),
    "v2": dict(
        bearish=BEARISH_V2,
        bullish=BULLISH_V2,
        idiom=IDIOM_V2,
        subjek=SUBJEK_V12,
        subjek_pola=None,
        numerik=True,
        flip_negasi=False,
        elongasi=False,
        negasi_kalimat=False,
        pola_minus=POLA_MINUS,
    ),
    "v3": dict(
        bearish=BEARISH_V3,
        bullish=BULLISH_V3,
        idiom=IDIOM_V3,
        subjek=SUBJEK_V3,
        subjek_pola=None,
        numerik=True,
        flip_negasi=True,
        elongasi=True,
        negasi_kalimat=False,
        pola_minus=POLA_MINUS,
    ),
    "v4": dict(
        bearish=BEARISH_V3,
        bullish=BULLISH_V3,
        idiom=IDIOM_V3,
        subjek=None,
        subjek_pola=SUBJEK_V45_POLA,
        numerik=True,
        flip_negasi=True,
        elongasi=True,
        negasi_kalimat=True,
        pola_minus=POLA_MINUS_V45,
    ),
    "v5": dict(
        bearish=BEARISH_V5,
        bullish=BULLISH_V5,
        idiom=IDIOM_V3,
        subjek=None,
        subjek_pola=SUBJEK_V45_POLA,
        numerik=True,
        flip_negasi=True,
        elongasi=True,
        negasi_kalimat=True,
        pola_minus=POLA_MINUS_V45,
    ),
}
KOLOM_HYBRID = {
    "v1": "hybrid",
    "v2": "hybrid2",
    "v3": "hybrid3",
    "v4": "hybrid4",
    "v5": "hybrid5",
}

# Pra-kompilasi pola cue agar loop utama cepat
for _v, _cfg in VERSI.items():
    _cfg["re_bearish"] = [re.compile(rf"\b{re.escape(c)}\b") for c in _cfg["bearish"]]
    _cfg["re_bullish"] = [re.compile(rf"\b{re.escape(c)}\b") for c in _cfg["bullish"]]
    _cfg["re_subjek"] = (
        [re.compile(p) for p in _cfg["subjek_pola"]]
        if _cfg["subjek_pola"] is not None
        else [re.compile(rf"\b{re.escape(s)}\b") for s in _cfg["subjek"]]
    )


# ==================================================================
# INTI LEKSIKON
# ==================================================================
def _token_sebelum(
    t: str, start: int, n: int = 2, stop_kalimat: bool = False
) -> List[str]:
    potong = t[:start]
    if stop_kalimat:
        potong = re.split(r"[.!?;,]", potong)[-1]
    return re.findall(r"[a-z0-9]+", potong)[-n:]


def label_leksikon(text: str, cfg: Dict) -> Optional[str]:
    t = str(text).lower()
    if cfg["elongasi"]:
        t = re.sub(r"(.)\1{2,}", r"\1", t)  # "turuuun" -> "turun"
    for idiom in cfg["idiom"]:
        t = t.replace(idiom, " ")

    if not any(p.search(t) for p in cfg["re_subjek"]):
        return None

    def hitung(pola_list):
        hit, ternegasi = 0, 0
        for pola in pola_list:
            for m in pola.finditer(t):
                tok = _token_sebelum(t, m.start(), stop_kalimat=cfg["negasi_kalimat"])
                if set(tok) & NEGATOR:
                    ternegasi += 1
                else:
                    hit += 1
        return hit, ternegasi

    bear, _ = hitung(cfg["re_bearish"])  # bearish ternegasi dibuang
    bull, bull_neg = hitung(cfg["re_bullish"])
    if cfg["flip_negasi"]:
        bear += bull_neg  # "gagal menguat" = bearish

    if cfg["numerik"]:
        bear += len(cfg["pola_minus"].findall(t))
        bull += len(POLA_PLUS.findall(t))

    if bear > bull:
        return "neg"
    if bull > bear:
        return "pos"
    return None


# ==================================================================
# TAHAP K: FILTER + LEKSIKON + HYBRID (SEKALI JALAN)
# ==================================================================
def tahap_k() -> None:
    if not FILE_INPUT.exists():
        raise FileNotFoundError(
            f"{FILE_INPUT} tidak ditemukan. Letakkan file ini satu folder "
            f"dengan dataset_multilabel.csv hasil fase2b."
        )
    df = pd.read_csv(FILE_INPUT)
    wajib = [TEXT_COL, DATE_COL] + [f"label_{a}" for a in ALIAS_MODEL]
    hilang = [c for c in wajib if c not in df.columns]
    if hilang:
        raise KeyError(
            f"Kolom {hilang} tidak ada di {FILE_INPUT}. Jalankan fase2b Tahap A dulu."
        )
    lap: list = ["LAPORAN FASE 2H (FILTER + LEKSIKON V1-V5 + HYBRID)", "=" * 60]
    n0 = len(df)
    logger.info(f"Memuat {n0} baris dari {FILE_INPUT}")

    # ---------- LANGKAH 1: filter relevansi ----------
    t = df[TEXT_COL].astype(str).str.lower()
    for nama, pola in {**KATEGORI_EKSKLUSI, **KATEGORI_LAPOR}.items():
        df[f"flag_{nama}"] = t.str.contains(pola, na=False)
    df["punya_jangkar_id"] = t.str.contains(POLA_JANGKAR_ID, na=False)
    ada_asing = df[[f"flag_{k}" for k in KATEGORI_EKSKLUSI]].any(axis=1)
    df["eksklusi_manual"] = (
        df["id"].isin(EKSKLUSI_MANUAL_IDS) if "id" in df.columns else False
    )
    df["eksklusi_konteks"] = (ada_asing & ~df["punya_jangkar_id"]) | df[
        "eksklusi_manual"
    ]

    tgl = pd.to_datetime(df[DATE_COL], errors="coerce")
    lap.append(f"Rentang tanggal data : {tgl.min().date()} s.d. {tgl.max().date()}")
    di_luar = int(
        (
            (tgl < pd.Timestamp(EVENT_WINDOW[0]))
            | (tgl > pd.Timestamp(EVENT_WINDOW[1]))
        ).sum()
    )
    if di_luar:
        lap.append(
            f"PERINGATAN: {di_luar} tweet di luar jendela Bab III "
            f"{EVENT_WINDOW}. Konfirmasi jendela final sebelum NSt."
        )

    lap.append("")
    lap.append(f"Total tweet masuk: {n0}")
    for nama in KATEGORI_EKSKLUSI:
        n_f = int(df[f"flag_{nama}"].sum())
        n_t = int((df[f"flag_{nama}"] & df["punya_jangkar_id"]).sum())
        lap.append(
            f"  {nama:24s}: {n_f:5d} terdeteksi, "
            f"{n_t:4d} dipertahankan (jangkar Indonesia)"
        )
    for nama in KATEGORI_LAPOR:
        lap.append(
            f"  {nama + ' (lapor saja)':24s}: "
            f"{int(df[f'flag_{nama}'].sum()):5d} terdeteksi, tidak dieksklusi"
        )
    n_ekskl = int(df["eksklusi_konteks"].sum())
    lap.append(
        f"Tweet dikeluarkan (termasuk {int(df['eksklusi_manual'].sum())} "
        f"putusan manual) : {n_ekskl}"
    )
    lap.append(f"Tweet dipertahankan : {n0 - n_ekskl}")
    lap.append(
        "Baris untuk Tabel 4.1 Bab IV: 'Konten di luar konteks pasar "
        f"modal Indonesia' : {n_ekskl}"
    )

    df[df["eksklusi_konteks"]].to_csv(FILE_EKSKLUSI, index=False, encoding="utf-8-sig")
    kolom_flag = [f"flag_{k}" for k in {**KATEGORI_EKSKLUSI, **KATEGORI_LAPOR}] + [
        "punya_jangkar_id",
        "eksklusi_manual",
        "eksklusi_konteks",
    ]
    df = df[~df["eksklusi_konteks"]].drop(columns=kolom_flag).reset_index(drop=True)

    # ---------- LANGKAH 2: leksikon V1-V5 ----------
    for v in HITUNG_VERSI:
        logger.info(f"Menghitung leksikon {v}...")
        df[f"leks_{v}"] = df[TEXT_COL].map(lambda x, _v=v: label_leksikon(x, VERSI[_v]))

    lap.append("")
    lap.append("=== CAKUPAN LEKSIKON ===")
    for v in HITUNG_VERSI:
        n = int(df[f"leks_{v}"].notna().sum())
        lap.append(
            f"  {v}: {n:5d} tweet bersinyal | "
            f"{df[f'leks_{v}'].value_counts().to_dict()}"
        )

    # Bukti kata tambahan V5 (untuk lampiran audit)
    if {"v4", "v5"} <= set(HITUNG_VERSI):
        t2 = df[TEXT_COL].astype(str).str.lower()
        baru = df["leks_v5"].notna() & df["leks_v4"].isna()
        lap.append("")
        lap.append(f"=== BUKTI TAMBAHAN V5 (tangkapan baru: {int(baru.sum())}) ===")
        for arah, daftar in [
            ("BEARISH", TAMBAHAN_BEARISH_V5),
            ("BULLISH", TAMBAHAN_BULLISH_V5),
        ]:
            for kata in daftar:
                m = t2.str.contains(rf"\b{re.escape(kata)}\b", na=False)
                lap.append(
                    f"  [{arah}] {kata:13s}: {int(m.sum()):4d} kemunculan, "
                    f"{int((m & baru).sum()):3d} tangkapan baru"
                )
                for _, r in df[m & baru].head(3).iterrows():
                    lap.append(f"      [{r['leks_v5']}] {str(r[TEXT_COL])[:95]}")

    # ---------- LANGKAH 3: kolom hybrid ----------
    lap.append("")
    for alias in ALIAS_MODEL:
        base = df[f"label_{alias}"]
        boleh = (
            base.eq("neu") if OVERRIDE_HANYA_NEU else pd.Series(True, index=df.index)
        )
        u = {}
        for v in HITUNG_VERSI:
            kol = f"label_{KOLOM_HYBRID[v]}_{alias}"
            df[kol] = np.where(df[f"leks_{v}"].notna() & boleh, df[f"leks_{v}"], base)
            u[v] = int((df[kol] != base).sum())
        pesan = f"[{alias}] koreksi " + " ".join(f"{v}={u[v]}" for v in HITUNG_VERSI)
        logger.info(pesan)
        lap.append(pesan)

    # Penambang kata pada residu (sumber sah penambahan berikutnya,
    # tetapi leksikon sudah BEKU; simpan hanya sebagai dokumentasi)
    v_akhir = HITUNG_VERSI[-1]
    sisa = (
        df[df[f"label_{KOLOM_HYBRID[v_akhir]}_mdhugol"].eq("neu")][TEXT_COL]
        .astype(str)
        .str.lower()
    )
    sisa = sisa[sisa.str.contains("ihsg|saham|bursa", regex=True, na=False)]
    stop = set(
        "yang dan di ke dari ini itu dengan untuk pada juga ada tidak akan "
        "bisa sudah masih hari the of to in and ihsg saham bursa pasar "
        "indeks".split()
    )
    tokens = Counter(
        w for teks in sisa for w in re.findall(r"[a-z]{4,}", teks) if w not in stop
    )
    lap.append("")
    lap.append("=== 25 KATA TERSERING DI RESIDU NEU (dokumentasi; leksikon beku) ===")
    for w, c in tokens.most_common(25):
        lap.append(f"  {w:20s} {c}")

    df.to_csv(FILE_FINAL, index=False, encoding="utf-8-sig")
    Path(FILE_LAPORAN).write_text("\n".join(lap), encoding="utf-8")
    print("\n".join(lap))
    logger.info(f"Tersimpan: {FILE_FINAL} ({len(df)} baris)")
    logger.info(f"Tersimpan: {FILE_EKSKLUSI} ({n_ekskl} baris)")
    logger.info(f"Tersimpan: {FILE_LAPORAN}")
    logger.info(
        "Berikutnya: tarik gold sample dari dataset_multilabel_final.csv, "
        "label manual ke sampel_validasi_manual.csv, lalu TAHAP='E'."
    )


# ==================================================================
# TAHAP E: EVALUASI SEMUA PELABEL VS LABEL MANUAL
# ==================================================================
def tahap_e() -> None:
    if not FILE_FINAL.exists():
        raise FileNotFoundError(f"{FILE_FINAL} belum ada. Jalankan TAHAP='K' dulu.")
    if not FILE_GOLD.exists():
        raise FileNotFoundError(
            f"{FILE_GOLD} belum ada. Tarik sampel dari {FILE_FINAL} "
            f"(kolom row_key + label_manual berisi pos/neu/neg)."
        )
    try:
        from sklearn.metrics import accuracy_score, classification_report, f1_score
    except ImportError as e:
        raise ImportError(
            "scikit-learn belum terpasang. Jalankan: pip install scikit-learn"
        ) from e
    multi = pd.read_csv(FILE_FINAL)
    gold = pd.read_csv(FILE_GOLD)
    gold["label_manual"] = gold["label_manual"].astype(str).str.strip().str.lower()
    gold = gold[gold["label_manual"].isin(VALID_CLS)]
    if len(gold) < 30:
        raise ValueError(f"Baru {len(gold)} baris gold valid. Lengkapi dulu.")
    logger.info(f"Gold sample terpakai: {len(gold)} baris.")

    merged = gold.merge(multi, on="row_key", how="inner", suffixes=("", "_m"))
    kandidat = [f"label_{a}" for a in ALIAS_MODEL]
    for v in HITUNG_VERSI:
        kandidat += [f"label_{KOLOM_HYBRID[v]}_{a}" for a in ALIAS_MODEL]
    kandidat = [k for k in kandidat if k in merged.columns]

    hasil = []
    for kol in kandidat:
        nama = kol.replace("label_", "", 1)
        y_true, y_pred = merged["label_manual"], merged[kol]
        acc = accuracy_score(y_true, y_pred)
        f1m = f1_score(y_true, y_pred, average="macro")
        hasil.append((nama, acc, f1m))
        print(f"\n===== {nama} =====")
        print(f"Akurasi: {acc:.3f} | Macro-F1: {f1m:.3f}")
        print(classification_report(y_true, y_pred, zero_division=0))

    hasil.sort(key=lambda x: x[2], reverse=True)
    print("\nPERINGKAT (Macro-F1):")
    for nama, acc, f1m in hasil:
        print(f"  {nama:26s} macro-F1={f1m:.3f}  akurasi={acc:.3f}")
    print(f"\nPemenang: '{hasil[0][0]}'. Isi MODEL_FINAL lalu TAHAP = 'F'.")


# ==================================================================
# TAHAP F: FINALISASI DATASET & NST
# ==================================================================
def tahap_f() -> None:
    pilihan = list(ALIAS_MODEL) + [
        f"{KOLOM_HYBRID[v]}_{a}" for v in HITUNG_VERSI for a in ALIAS_MODEL
    ]
    if MODEL_FINAL not in pilihan:
        raise ValueError(
            f"MODEL_FINAL='{MODEL_FINAL}' tidak valid. "
            f"Pilihan: {pilihan}. Jalankan Tahap E dulu."
        )
    df = pd.read_csv(FILE_FINAL)
    df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors="coerce")
    df = df.dropna(subset=[DATE_COL])
    df["label_sent"] = df[f"label_{MODEL_FINAL}"]

    scored = OUTPUT_FOLDER / "dataset_scored.csv"
    df.to_csv(scored, index=False, encoding="utf-8-sig")
    logger.info(f"Tersimpan: {scored} (pelabel final: {MODEL_FINAL})")

    # Catatan Bab IV: dengan label diskret, S_i = +1/0/-1 sehingga
    # NSt = (n_pos - n_neg)/n_total adalah bentuk diskret rumus Bab III.
    daily = (
        df.groupby(df[DATE_COL].dt.date)["label_sent"]
        .value_counts()
        .unstack(fill_value=0)
        .reset_index()
    )
    daily = daily.rename(columns={DATE_COL: "tanggal"})
    daily.columns.name = None
    for col in ("pos", "neu", "neg"):
        if col not in daily.columns:
            daily[col] = 0
    daily["n_total"] = daily[["pos", "neu", "neg"]].sum(axis=1)
    daily["NSt"] = np.where(
        daily["n_total"] > 0, (daily["pos"] - daily["neg"]) / daily["n_total"], 0.0
    )
    daily = daily.sort_values("tanggal").reset_index(drop=True)
    daily = daily[["tanggal", "pos", "neu", "neg", "n_total", "NSt"]]

    nst = OUTPUT_FOLDER / "nst_harian.csv"
    daily.to_csv(nst, index=False, encoding="utf-8-sig")
    logger.info(
        f"Tersimpan: {nst} | {len(daily)} hari | pelabel={MODEL_FINAL} | "
        f"NSt min={daily['NSt'].min():.4f}, max={daily['NSt'].max():.4f}"
    )


# ==================================================================
# ENTRY POINT
# ==================================================================
if __name__ == "__main__":
    try:
        if TAHAP == "K":
            tahap_k()
        elif TAHAP == "E":
            tahap_e()
        elif TAHAP == "F":
            tahap_f()
        else:
            raise ValueError("TAHAP harus 'K', 'E', atau 'F'.")
    except Exception as e:
        logger.error(f"Berhenti: {e}", exc_info=True)
        raise

<>:49: SyntaxWarning: invalid escape sequence '\M'
<>:50: SyntaxWarning: invalid escape sequence '\M'
<>:51: SyntaxWarning: invalid escape sequence '\M'
<>:52: SyntaxWarning: invalid escape sequence '\M'
<>:49: SyntaxWarning: invalid escape sequence '\M'
<>:50: SyntaxWarning: invalid escape sequence '\M'
<>:51: SyntaxWarning: invalid escape sequence '\M'
<>:52: SyntaxWarning: invalid escape sequence '\M'
C:\Users\EZRA\AppData\Local\Temp\ipykernel_3236\521688632.py:49: SyntaxWarning: invalid escape sequence '\M'
  FILE_FINAL = OUTPUT_FOLDER / "G:\My Drive\skripsi\proje\proje\data\interim\dataset_multilabel_final.csv"
C:\Users\EZRA\AppData\Local\Temp\ipykernel_3236\521688632.py:50: SyntaxWarning: invalid escape sequence '\M'
  FILE_EKSKLUSI = OUTPUT_FOLDER / "G:\My Drive\skripsi\proje\proje\data\interim\dataset_eksklusi_konteks.csv"
C:\Users\EZRA\AppData\Local\Temp\ipykernel_3236\521688632.py:51: SyntaxWarning: invalid escape sequence '\M'
  FILE_LAPORAN = OUTPUT_FOLDER / "G:\My Drive\sk

LAPORAN FASE 2H (FILTER + LEKSIKON V1-V5 + HYBRID)
Rentang tanggal data : 2025-06-01 s.d. 2025-10-31
PERINGATAN: 1223 tweet di luar jendela Bab III ('2025-03-01', '2025-09-30'). Konfirmasi jendela final sebelum NSt.

Total tweet masuk: 14050
  pasar_luar_negeri       :   191 terdeteksi,  191 dipertahankan (jangkar Indonesia)
  kripto                  :     7 terdeteksi,    7 dipertahankan (jangkar Indonesia)
  bursa_non_pasar_modal   :     6 terdeteksi,    6 dipertahankan (jangkar Indonesia)
  bahasa_melayu_malaysia  :    59 terdeteksi,   59 dipertahankan (jangkar Indonesia)
  saham_metafora_pribadi (lapor saja):     2 terdeteksi, tidak dieksklusi
  komoditas (lapor saja)  :    59 terdeteksi, tidak dieksklusi
Tweet dikeluarkan (termasuk 0 putusan manual) : 0
Tweet dipertahankan : 14050
Baris untuk Tabel 4.1 Bab IV: 'Konten di luar konteks pasar modal Indonesia' : 0

=== CAKUPAN LEKSIKON ===
  v1:  3454 tweet bersinyal | {'pos': 2052, 'neg': 1402}
  v2:  4696 tweet bersinyal | {'pos': 2

Labeling dan visualisasi data 

In [ ]:
import logging
from contextlib import contextmanager
from pathlib import Path
from typing import Optional, List, Dict, Generator, Any, Callable

import pandas as pd
import matplotlib

# Backend non-interaktif absolut: aman untuk server headless, cron job, & batch processing
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# =============================================================================
# 0. KONFIGURASI GLOBAL & ESTETIKA MURNI
# =============================================================================

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - [%(funcName)s] - %(message)s",
)
logger = logging.getLogger(__name__)

# Mematikan GUI event loop untuk menghemat RAM dan siklus CPU
plt.ioff()

plt.rcParams.update(
    {
        "figure.figsize": (12, 6),
        "font.family": "serif",
        "font.size": 11,
        "axes.titlesize": 14,
        "axes.labelsize": 12,
        "savefig.bbox": "tight",
    }
)
plt.style.use("seaborn-v0_8-whitegrid")

COLORS: Dict[str, str] = {"pos": "#2ecc71", "neu": "#95a5a6", "neg": "#e74c3c"}
LABELS: Dict[str, str] = {"pos": "Positif", "neu": "Netral", "neg": "Negatif"}
SENTIMENT_COLS: List[str] = ["pos", "neu", "neg"]


# =============================================================================
# 1. DATA ACCESS LAYER (DAL) - HIGH PERFORMANCE I/O & AUTOMATED AGGREGATION
# =============================================================================
def load_and_sanitize_data(file_path: Path) -> pd.DataFrame:
    """Membaca CSV mentah, melakukan agregasi harian, dan menghitung otomatis NSt."""
    if not file_path.exists():
        raise FileNotFoundError(
            f"Integritas I/O gagal. File tidak ditemukan: {file_path}"
        )

    # Membaca langsung jalur path untuk mengaktifkan memory-mapping OS
    df_raw = pd.read_csv(
        file_path, engine="pyarrow", dtype_backend="pyarrow", encoding_errors="replace"
    )

    if df_raw.empty:
        raise ValueError("Dataset dimuat dengan sukses tetapi kosong (0 baris).")

    if "tanggal_wib" not in df_raw.columns:
        raise KeyError(
            "Skema rusak. Kolom 'tanggal_wib' tidak ditemukan di dataset mentah."
        )

    # Ekstrak tanggal saja (buang jam/menit/detik) untuk pengelompokan harian
    df_raw["tanggal"] = pd.to_datetime(
        df_raw["tanggal_wib"], format="mixed", errors="coerce"
    ).dt.date
    df_raw = df_raw.dropna(subset=["tanggal"])

    # Menggunakan acuan model 'label_mdhugol' untuk agregasi teks mentah ke hitungan harian
    kolom_label = "label_mdhugol"
    if kolom_label not in df_raw.columns:
        raise KeyError(f"Skema rusak. Kolom model {kolom_label} tidak ditemukan.")

    df_agg = df_raw.groupby(["tanggal", kolom_label]).size().unstack(fill_value=0)

    # Memastikan kolom pos, neu, neg selalu ada meskipun di tanggal tertentu bernilai 0
    for col in ["pos", "neu", "neg"]:
        if col not in df_agg.columns:
            df_agg[col] = 0

    df_agg = df_agg.reset_index()

    # Kalkulasi Net Sentiment Index (NSt) secara otomatis
    total_volume = df_agg["pos"] + df_agg["neu"] + df_agg["neg"]
    df_agg["NSt"] = (df_agg["pos"] - df_agg["neg"]) / total_volume.replace(0, 1)

    # Format kembali ke tipe datetime standar Matplotlib
    df_agg["tanggal"] = pd.to_datetime(df_agg["tanggal"])

    logger.info(
        f"Transformasi DAL Sukses: {len(df_agg)} entri harian unik berhasil diagregasi dari data mentah."
    )
    return df_agg.sort_values("tanggal").reset_index(drop=True)


# =============================================================================
# 2. PRESENTATION LAYER - ISOLASI SIKLUS HIDUP MEMORI
# =============================================================================
@contextmanager
def managed_figure() -> Generator[plt.Figure, None, None]:
    """Isolator mutlak memori RAM untuk mencegah memory leak pada Matplotlib."""
    fig = plt.figure()
    try:
        yield fig
    finally:
        fig.clf()
        plt.close(fig)


# =============================================================================
# 3. PURE PLOTTING FUNCTIONS (NATIVE C-API RENDERING)
# =============================================================================
def plot_pie_distribution(df: pd.DataFrame, ax: plt.Axes) -> None:
    """Menghasilkan pie chart proporsi sentimen secara keseluruhan."""
    total = df[SENTIMENT_COLS].sum()
    if total.sum() == 0:
        raise ValueError("Total agregat sentimen bernilai absolut 0.")

    ax.pie(
        total,
        labels=[LABELS[c] for c in total.index],
        colors=[COLORS[c] for c in total.index],
        autopct="%1.1f%%",
        startangle=140,
        wedgeprops={"edgecolor": "white", "linewidth": 1.5},
    )
    ax.set_title("Distribusi Total Sentimen", fontweight="bold")


def plot_nst_with_metric(
    df: pd.DataFrame,
    ax: plt.Axes,
    date_col: str = "tanggal",
    nst_col: str = "NSt",
    metric_col: Optional[str] = None,
    metric_label: str = "Variabel Dependen",
) -> None:
    """Line chart NSt harian dengan dukungan dual-axis opsional."""
    plot_df = df.dropna(subset=[nst_col])
    if plot_df.empty:
        raise ValueError(f"Kolom target ({nst_col}) sepenuhnya kosong atau NaN.")

    ax.plot(
        plot_df[date_col],
        plot_df[nst_col],
        color="#2980b9",
        linewidth=2.5,
        marker="o",
        markersize=6,
        label="Net Sentiment Index (NSt)",
    )
    ax.axhline(
        0,
        color="#c0392b",
        linestyle="--",
        linewidth=1.5,
        alpha=0.8,
        label="Ambang Netral",
    )
    ax.set_ylabel("Indeks Sentimen (-1.0 s.d. 1.0)")

    lines, labels = ax.get_legend_handles_labels()

    if metric_col and metric_col in plot_df.columns:
        metric_df = plot_df.dropna(subset=[metric_col])
        if not metric_df.empty:
            ax2 = ax.twinx()
            ax2.plot(
                metric_df[date_col],
                metric_df[metric_col],
                color="#e67e22",
                linewidth=2,
                linestyle="-.",
                marker="s",
                markersize=6,
                label=metric_label,
            )
            ax2.set_ylabel(metric_label)

            lines2, labels2 = ax2.get_legend_handles_labels()
            lines.extend(lines2)
            labels.extend(labels2)
            ax2.get_legend().remove()
    elif metric_col:
        logger.warning(f"Metrik '{metric_col}' tidak ditemukan, merender NSt tunggal.")

    ax.legend(lines, labels, loc="upper left", frameon=True, shadow=True)

    ax.xaxis.set_major_formatter(mdates.DateFormatter("%d %b '%y"))
    ax.xaxis.set_major_locator(mdates.AutoDateLocator())
    plt.setp(ax.get_xticklabels(), rotation=30, ha="right")
    ax.set_xlabel("Periode Perdagangan")


def plot_stacked_volume(
    df: pd.DataFrame, ax: plt.Axes, date_col: str = "tanggal"
) -> None:
    """Stacked bar volume harian menggunakan rendering native Matplotlib."""
    dates = df[date_col]
    pos = df["pos"]
    neu = df["neu"]
    neg = df["neg"]

    ax.bar(dates, pos, label=LABELS["pos"], color=COLORS["pos"], width=0.8)
    ax.bar(dates, neu, bottom=pos, label=LABELS["neu"], color=COLORS["neu"], width=0.8)
    ax.bar(
        dates,
        neg,
        bottom=pos + neu,
        label=LABELS["neg"],
        color=COLORS["neg"],
        width=0.8,
    )

    ax.set_title("Kepadatan Volume Sentimen Harian", fontweight="bold")
    ax.set_xlabel("Periode Perdagangan")
    ax.set_ylabel("Frekuensi Dokumen/Teks")
    ax.legend(title="Kelas Sentimen", loc="upper right")

    ax.xaxis.set_major_formatter(mdates.DateFormatter("%d %b '%y"))
    ax.xaxis.set_major_locator(mdates.AutoDateLocator())
    plt.setp(ax.get_xticklabels(), rotation=30, ha="right")


# =============================================================================
# 4. ORKESTRATOR PIPELINE
# =============================================================================
def eksekusi_pipeline_visualisasi(
    file_harian: str,
    metric_col: Optional[str] = None,
    metric_label: str = "Metrik Dependen",
    output_dir: str = ".",
    format_gambar: str = "png",
    dpi_gambar: int = 300,
) -> None:
    """Pipeline orkestrator utama untuk mengeksekusi ekstraksi dan rendering visual."""
    out_path = Path(output_dir)
    out_path.mkdir(parents=True, exist_ok=True)

    try:
        df = load_and_sanitize_data(Path(file_harian))
    except Exception as e:
        logger.critical(f"Pipeline gagal pada Data Access Layer: {e}")
        return

    plot_tasks: List[tuple[Callable, dict[str, Any], str, str]] = [
        (plot_pie_distribution, {"df": df}, "1_distribusi_pie", "Pie Chart"),
        (
            plot_nst_with_metric,
            {"df": df, "metric_col": metric_col, "metric_label": metric_label},
            "2_pergerakan_nst",
            "Line Chart NSt",
        ),
        (plot_stacked_volume, {"df": df}, "3_volume_harian", "Stacked Bar Volume"),
    ]

    for plot_func, kwargs, file_basename, name in plot_tasks:
        try:
            with managed_figure() as fig:
                ax = fig.subplots()
                kwargs["ax"] = ax
                plot_func(**kwargs)

                if name == "Line Chart NSt":
                    fig.suptitle(
                        "Dinamika Net Sentiment vs Pergerakan Pasar",
                        fontweight="bold",
                        y=0.98,
                    )

                filename = f"{file_basename}.{format_gambar.lower()}"
                fig.savefig(out_path / filename, dpi=dpi_gambar, format=format_gambar)
                logger.info(f"Aset berhasil disimpan: {filename}")
        except Exception as e:
            logger.error(f"Gagal merender modul {name}: {e}")

    logger.info(f"Pipeline selesai. Seluruh aset tersimpan di: {out_path.resolve()}")


# =============================================================================
# 5. ENTRY POINT
# =============================================================================
if __name__ == "__main__":
    # Gunakan 'r' (Raw String) untuk path Windows agar backslash (\) aman
    FILE_INPUT = r"C:\Users\EZRA\Downloads\Programs\dataset_multilabel_final.csv"
    FOLDER_OUTPUT = r"G:\My Drive\skripsi\proje\proje\data\hasil_visualisasi"

    eksekusi_pipeline_visualisasi(
        file_harian=FILE_INPUT,
        output_dir=FOLDER_OUTPUT,
        format_gambar="png",
        dpi_gambar=300,
    )

2026-07-20 13:01:54,723 - INFO - [load_and_sanitize_data] - Transformasi DAL Sukses: 153 entri harian unik berhasil diagregasi dari data mentah.
2026-07-20 13:01:54,876 - INFO - [eksekusi_pipeline_visualisasi] - Aset berhasil disimpan: 1_distribusi_pie.png
2026-07-20 13:01:55,159 - INFO - [eksekusi_pipeline_visualisasi] - Aset berhasil disimpan: 2_pergerakan_nst.png
2026-07-20 13:01:55,699 - INFO - [eksekusi_pipeline_visualisasi] - Aset berhasil disimpan: 3_volume_harian.png
2026-07-20 13:01:55,708 - INFO - [eksekusi_pipeline_visualisasi] - Pipeline selesai. Seluruh aset tersimpan di: G:\My Drive\skripsi\proje\proje\data\hasil_visualisasi


In [ ]:
Statistikan data harian dari dataset_multilabel_final.csv, buat visualisasi pie chart distribusi sentimen, line chart NSt, dan stacked bar volume harian


In [ ]:
"""
Enterprise-Grade Sentiment Analysis Statistical Pipeline (The Pinnacle)
Target Hardware: Intel Core i7-12th Gen (PyArrow Vectorization) + NVIDIA RTX 4050
Backend: Pure OO Matplotlib (Zero pyplot, Memory-leak Protected, Fully Typed)
"""

import logging
import warnings
from contextlib import contextmanager
from dataclasses import dataclass, asdict
from enum import Enum, unique
from pathlib import Path
from typing import Optional, Dict, List, Callable, Generator, Tuple, Any

import numpy as np
import pandas as pd
from scipy import stats
from scipy.stats import gaussian_kde

import matplotlib

matplotlib.use("Agg")  # Backend non-GUI absolut untuk Server/Batch Execution
from matplotlib.figure import Figure
from matplotlib.backends.backend_agg import FigureCanvasAgg
from matplotlib.axes import Axes
import matplotlib.dates as mdates

# =============================================================================
# 0. GLOBAL CONFIGURATION, LOGGER & DOMAIN ENUMS
# =============================================================================
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - [%(funcName)s] - %(message)s",
)
logger = logging.getLogger(__name__)
warnings.filterwarnings("ignore", category=RuntimeWarning)


@unique
class Sentiment(Enum):
    POS = "pos"
    NEU = "neu"
    NEG = "neg"


# Separation of Concerns: Domain Colors vs UI Accent Palette
COLORS: Dict[Sentiment, str] = {
    Sentiment.POS: "#2ecc71",
    Sentiment.NEU: "#95a5a6",
    Sentiment.NEG: "#e74c3c",
}

COLOR_PALETTE: Dict[str, str] = {
    "primary": "#2980b9",
    "secondary": "#8e44ad",
    "danger": "#c0392b",
    "warning": "#e67e22",
    "neutral": "#7f8c8d",
}

LABELS: Dict[Sentiment, str] = {
    Sentiment.POS: "Positif",
    Sentiment.NEU: "Netral",
    Sentiment.NEG: "Negatif",
}

SENTIMENT_COLS: List[str] = [s.value for s in Sentiment]

# Matplotlib Global rcParams Styling
matplotlib.rcParams.update(
    {
        "figure.figsize": (12, 6),
        "font.family": "serif",
        "font.size": 11,
        "savefig.bbox": "tight",
        "axes.titlesize": 14,
        "axes.grid": True,
        "grid.alpha": 0.5,
    }
)


# =============================================================================
# 1. IMMUTABLE DOMAIN ENTITIES (DTOs - RAM Optimized via slots)
# =============================================================================
@dataclass(frozen=True, slots=True)
class DistributionSummary:
    kelas: str
    jumlah: int
    persentase: float

    def to_dict(self) -> Dict[str, object]:
        return asdict(self)


@dataclass(frozen=True, slots=True)
class NStDescriptiveStats:
    mean: float
    median: float
    std_dev: float
    minimum: float
    maximum: float
    skewness: float
    kurtosis: float
    q1: float
    q3: float
    iqr: float
    ci_lower_95: float
    ci_upper_95: float

    def to_dict(self) -> Dict[str, object]:
        return asdict(self)


@dataclass(frozen=True, slots=True)
class CorrelationResult:
    pearson_r: float
    pearson_p: float
    spearman_r: float
    spearman_p: float
    interpretation: str

    def to_dict(self) -> Dict[str, object]:
        return asdict(self)


# =============================================================================
# 2. DATA ACCESS LAYER (PyArrow Engine - Hardware Optimized)
# =============================================================================
def load_and_sanitize_data(file_path: Path) -> pd.DataFrame:
    if not file_path.exists():
        raise FileNotFoundError(
            f"Integritas I/O gagal: File tidak ditemukan di {file_path}"
        )

    # PyArrow Backend untuk utilitasi multi-threading i7-12th Gen
    df_raw = pd.read_csv(
        file_path, engine="pyarrow", dtype_backend="pyarrow", encoding_errors="replace"
    )
    if df_raw.empty:
        raise ValueError("I/O sukses, namun dataset berukuran 0 baris.")

    # Proteksi konversi UTC ke LocalDate (WIB Asumsi) via ISO8601 parsing
    dt_series = pd.to_datetime(df_raw["tanggal_wib"], format="ISO8601", errors="coerce")
    df_raw["tanggal"] = dt_series.dt.date
    df_raw = df_raw.dropna(subset=["tanggal"])

    df_agg = df_raw.groupby(["tanggal", "label_mdhugol"]).size().unstack(fill_value=0)
    for col in SENTIMENT_COLS:
        if col not in df_agg.columns:
            df_agg[col] = 0

    df_agg = df_agg.reset_index()
    total_vol = (
        df_agg[SENTIMENT_COLS[0]]
        + df_agg[SENTIMENT_COLS[1]]
        + df_agg[SENTIMENT_COLS[2]]
    )
    df_agg["NSt"] = (
        df_agg[Sentiment.POS.value] - df_agg[Sentiment.NEG.value]
    ) / total_vol.replace(0, 1)
    df_agg["tanggal"] = pd.to_datetime(df_agg["tanggal"])

    logger.info(
        f"DAL: Ekstraksi {len(df_agg)} entri harian unik selesai (PyArrow Engine)."
    )
    return df_agg.sort_values("tanggal").reset_index(drop=True)


# =============================================================================
# 3. STATISTICAL ENGINE (Vectorized & Fallback Protected)
# =============================================================================
def compute_distribution(df: pd.DataFrame) -> List[DistributionSummary]:
    total = df[SENTIMENT_COLS].sum()
    grand = int(total.sum())
    if grand == 0:
        raise ValueError("Gagal komputasi: Agregat sentimen 0 mutlak.")

    results = []
    for c in SENTIMENT_COLS:
        try:
            label = LABELS[Sentiment(c)]
        except ValueError:
            label = c.capitalize()  # Robust fallback mechanism

        results.append(
            DistributionSummary(
                kelas=label,
                jumlah=int(total[c]),
                persentase=round((int(total[c]) / grand) * 100, 2),
            )
        )
    results.append(DistributionSummary(kelas="Total", jumlah=grand, persentase=100.0))
    return results


def compute_nst_stats(df: pd.DataFrame) -> NStDescriptiveStats:
    nst = df["NSt"].dropna()
    n = len(nst)
    if n < 2:
        raise ValueError(
            "Populasi data harian tidak memenuhi kriteria statistik parametrik (n < 2)."
        )

    mean, std = float(nst.mean()), float(nst.std(ddof=1))
    margin = stats.t.ppf(0.975, n - 1) * (std / np.sqrt(n)) if std > 0 else 0.0

    return NStDescriptiveStats(
        mean=round(mean, 4),
        median=round(float(nst.median()), 4),
        std_dev=round(std, 4),
        minimum=round(float(nst.min()), 4),
        maximum=round(float(nst.max()), 4),
        skewness=round(float(nst.skew()), 4),
        kurtosis=round(float(nst.kurtosis()), 4),
        q1=round(float(nst.quantile(0.25)), 4),
        q3=round(float(nst.quantile(0.75)), 4),
        iqr=round(float(nst.quantile(0.75) - nst.quantile(0.25)), 4),
        ci_lower_95=round(mean - margin, 4),
        ci_upper_95=round(mean + margin, 4),
    )


def compute_temporal(
    df: pd.DataFrame, sma_windows: Tuple[int, ...] = (7, 14, 30)
) -> pd.DataFrame:
    """Modifikasi In-Place Memory-Efficient dengan Mapping Enum."""
    for w in sma_windows:
        df[f"NSt_SMA_{w}"] = df["NSt"].rolling(window=w, min_periods=1).mean()

    df["NSt_EMA_14"] = df["NSt"].ewm(span=14, min_periods=1).mean()
    df["NSt_Volatility_7"] = df["NSt"].rolling(window=7, min_periods=1).std().fillna(0)

    label_map = {s.value: LABELS[s] for s in Sentiment}
    df["Dominant"] = df[SENTIMENT_COLS].idxmax(axis=1).map(label_map)

    iso = df["tanggal"].dt.isocalendar()
    df["tahun_minggu"] = iso.year.astype(str) + "-W" + iso.week.astype(str).str.zfill(2)
    return df


def compute_correlation(
    df: pd.DataFrame, metric_col: Optional[str]
) -> Optional[CorrelationResult]:
    if not metric_col or metric_col not in df.columns:
        return None

    aligned = df[["NSt", metric_col]].dropna()
    if len(aligned) < 3:
        logger.warning(
            f"Data irisan '{metric_col}' kurang dari 3 baris. Korelasi dibatalkan."
        )
        return None

    if aligned["NSt"].std() == 0 or aligned[metric_col].std() == 0:
        logger.warning(
            f"Varians '{metric_col}' atau NSt bernilai nol (Flatline). Korelasi mustahil dihitung."
        )
        return None

    pr, pp = stats.pearsonr(aligned["NSt"], aligned[metric_col])
    sr, sp = stats.spearmanr(aligned["NSt"], aligned[metric_col])

    strength = (
        "Sangat lemah"
        if abs(pr) < 0.1
        else "Lemah"
        if abs(pr) < 0.3
        else "Moderat"
        if abs(pr) < 0.5
        else "Kuat"
        if abs(pr) < 0.7
        else "Sangat kuat"
    )
    return CorrelationResult(
        pearson_r=round(pr, 4),
        pearson_p=round(pp, 4),
        spearman_r=round(sr, 4),
        spearman_p=round(sp, 4),
        interpretation=f"{strength}-{'positif' if pr > 0 else 'negatif'}",
    )


# =============================================================================
# 4. PRESENTATION LAYER (Pure Object-Oriented Matplotlib)
# =============================================================================
@contextmanager
def managed_figure() -> Generator[Tuple[Figure, Axes], None, None]:
    """Manajemen memori absolut untuk server headless. Melindungi heap memory."""
    fig = Figure()
    FigureCanvasAgg(fig)
    ax = fig.add_subplot(111)
    try:
        yield fig, ax
    finally:
        fig.clear()


def _format_date_axis(ax: Axes) -> None:
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%d %b '%y"))
    ax.xaxis.set_major_locator(mdates.AutoDateLocator())
    for label in ax.get_xticklabels():
        label.set_rotation(30)
        label.set_horizontalalignment("right")


def plot_pie(df: pd.DataFrame, ax: Axes) -> None:
    total = df[SENTIMENT_COLS].sum()
    labels = [LABELS[Sentiment(c)] for c in total.index if c in SENTIMENT_COLS]
    colors = [COLORS[Sentiment(c)] for c in total.index if c in SENTIMENT_COLS]

    ax.pie(
        total.values,
        labels=labels,
        colors=colors,
        autopct="%1.1f%%",
        startangle=140,
        wedgeprops={"edgecolor": "white", "linewidth": 1.5},
    )
    ax.set_title("Distribusi Total Sentimen", fontweight="bold")


def plot_nst_line(
    df: pd.DataFrame, ax: Axes, metric_col: Optional[str] = None, metric_label: str = ""
) -> None:
    if not metric_label and metric_col:
        metric_label = metric_col

    plot_df = df.dropna(subset=["NSt"])
    ax.plot(
        plot_df["tanggal"],
        plot_df["NSt"],
        color=COLOR_PALETTE["primary"],
        linewidth=2.5,
        marker="o",
        markersize=5,
        label="NSt Harian",
    )
    ax.axhline(
        0,
        color=COLOR_PALETTE["danger"],
        linestyle="--",
        linewidth=1.2,
        alpha=0.8,
        label="Netral",
    )
    ax.set_ylabel("Indeks Sentimen (-1.0 s.d 1.0)")

    if (
        metric_col
        and metric_col in plot_df.columns
        and pd.api.types.is_numeric_dtype(plot_df[metric_col])
    ):
        ax2 = ax.twinx()
        ax2.plot(
            plot_df["tanggal"],
            plot_df[metric_col],
            color=COLOR_PALETTE["warning"],
            linewidth=2,
            linestyle="-.",
            marker="s",
            markersize=5,
            label=metric_label,
        )
        ax2.set_ylabel(metric_label)

        lines1, labels1 = ax.get_legend_handles_labels()
        lines2, labels2 = ax2.get_legend_handles_labels()
        ax.legend(lines1 + lines2, labels1 + labels2, loc="upper left", frameon=True)
        if ax2.get_legend():
            ax2.get_legend().remove()
    else:
        ax.legend(loc="upper left", frameon=True)

    _format_date_axis(ax)


def plot_stacked_volume(df: pd.DataFrame, ax: Axes) -> None:
    d, p, n, g = (
        df["tanggal"],
        df[Sentiment.POS.value],
        df[Sentiment.NEU.value],
        df[Sentiment.NEG.value],
    )
    ax.bar(d, p, label=LABELS[Sentiment.POS], color=COLORS[Sentiment.POS], width=0.8)
    ax.bar(
        d,
        n,
        bottom=p,
        label=LABELS[Sentiment.NEU],
        color=COLORS[Sentiment.NEU],
        width=0.8,
    )
    ax.bar(
        d,
        g,
        bottom=p + n,
        label=LABELS[Sentiment.NEG],
        color=COLORS[Sentiment.NEG],
        width=0.8,
    )
    ax.set_title("Volume Sentimen Harian", fontweight="bold")
    ax.set_ylabel("Frekuensi Dokumen")
    ax.legend(title="Kelas Sentimen", loc="upper right")
    _format_date_axis(ax)


def plot_histogram_nst(df: pd.DataFrame, ax: Axes) -> None:
    data = df["NSt"].dropna()
    ax.hist(
        data,
        bins="auto",
        color=COLOR_PALETTE["primary"],
        edgecolor="white",
        alpha=0.75,
        density=True,
    )
    kde = gaussian_kde(data)
    xs = np.linspace(data.min(), data.max(), 200)
    ax.plot(xs, kde(xs), color=COLOR_PALETTE["danger"], linewidth=2, label="KDE")
    ax.axvline(
        data.mean(),
        color=COLOR_PALETTE["warning"],
        linestyle="--",
        label=f"Mean: {data.mean():.3f}",
    )
    ax.set_xlabel("Net Sentiment Index (NSt)")
    ax.set_ylabel("Densitas Peluang")
    ax.set_title("Histogram & Distribusi KDE", fontweight="bold")
    ax.legend(loc="best", frameon=True)


def plot_heatmap_corr(df: pd.DataFrame, ax: Axes) -> None:
    extras = [
        c
        for c in df.columns
        if c not in SENTIMENT_COLS + ["NSt"] and pd.api.types.is_numeric_dtype(df[c])
    ][:3]
    all_cols = SENTIMENT_COLS + ["NSt"] + extras
    corr = df[all_cols].corr(method="pearson")

    im = ax.imshow(corr.values, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
    for i in range(len(corr)):
        for j in range(len(corr)):
            val = corr.iloc[i, j]
            ax.text(
                j,
                i,
                f"{val:.2f}",
                ha="center",
                va="center",
                color="white" if abs(val) > 0.6 else "black",
                fontsize=10,
            )

    ax.set_xticks(range(len(corr)))
    ax.set_yticks(range(len(corr)))
    ax.set_xticklabels(corr.columns, rotation=45, ha="right")
    ax.set_yticklabels(corr.columns)
    ax.set_title("Matriks Korelasi Pearson", fontweight="bold")
    ax.figure.colorbar(im, ax=ax, fraction=0.046, pad=0.04)


# =============================================================================
# 5. ORCHESTRATOR & EXPORT DISPATCHER
# =============================================================================
def export_reports(df: pd.DataFrame, out_dir: Path, metric_col: Optional[str]) -> None:
    out_dir.mkdir(parents=True, exist_ok=True)

    dist_df = pd.DataFrame([d.to_dict() for d in compute_distribution(df)])
    nst_df = pd.DataFrame([compute_nst_stats(df).to_dict()])
    corr = compute_correlation(df, metric_col)

    corr_df = (
        pd.DataFrame([corr.to_dict()])
        if corr
        else pd.DataFrame(
            {
                "info": [
                    "Korelasi dibatalkan: Data dependen tidak valid atau memiliki varians 0 mutlak."
                ]
            }
        )
    )

    with pd.ExcelWriter(
        out_dir / "ringkasan_statistik.xlsx", engine="openpyxl"
    ) as writer:
        dist_df.to_excel(writer, sheet_name="Distribusi", index=False)
        nst_df.to_excel(writer, sheet_name="Deskriptif_NSt", index=False)
        corr_df.to_excel(writer, sheet_name="Korelasi", index=False)

    dist_df.to_csv(
        out_dir / "tabel_distribusi_sentimen.csv",
        index=False,
        sep=";",
        encoding="utf-8-sig",
    )


def run_pipeline(
    file_input: str,
    output_dir: str,
    metric_col: Optional[str] = None,
    metric_label: str = "",
) -> None:
    """Master entry point – Command Registry Pattern, Strict Entity Flow."""
    out = Path(output_dir)
    df = compute_temporal(load_and_sanitize_data(Path(file_input)))
    export_reports(df, out, metric_col)

    if not metric_label and metric_col:
        metric_label = metric_col

    # O(1) Routing Execution
    tasks: Dict[str, Callable[[Axes], None]] = {
        "01_pie_distribusi": lambda ax: plot_pie(df, ax),
        "02_pergerakan_nst": lambda ax: plot_nst_line(df, ax, metric_col, metric_label),
        "03_stacked_volume": lambda ax: plot_stacked_volume(df, ax),
        "04_histogram_kde": lambda ax: plot_histogram_nst(df, ax),
        "05_heatmap_korelasi": lambda ax: plot_heatmap_corr(df, ax),
    }

    for basename, plot_fn in tasks.items():
        try:
            with managed_figure() as (fig, ax):
                plot_fn(ax)
                if "02_" in basename:
                    judul = (
                        "Dinamika Net Sentiment vs Pasar"
                        if metric_col
                        else "Dinamika Net Sentiment"
                    )
                    fig.suptitle(judul, fontweight="bold", y=0.98)
                fig.savefig(out / f"{basename}.png", dpi=300, format="png")
                logger.info(f"Render modul sukses: {basename}.png")
        except Exception as e:
            logger.error(
                f"Kegagalan fatal pada render modul [{basename}]: {str(e)}",
                exc_info=True,
            )

    logger.info(
        f"Siklus pipeline berakhir dengan integritas penuh. Destinasi: {out.resolve()}"
    )


if __name__ == "__main__":
    run_pipeline(
        file_input=r"G:\My Drive\skripsi\proje\proje\data\hasil_visualisasi\dataset_with_ihsg.csv",
        output_dir=r"G:\My Drive\skripsi\proje\proje\data\hasil_visualisasi",
        metric_col="harga_penutupan",
        metric_label="Harga Penutupan Saham",
    )

2026-07-20 22:08:05,676 - INFO - [load_and_sanitize_data] - DAL: Ekstraksi 153 entri harian unik selesai (PyArrow Engine).
2026-07-20 22:08:06,445 - INFO - [run_pipeline] - Render modul sukses: 01_pie_distribusi.png
2026-07-20 22:08:06,925 - INFO - [run_pipeline] - Render modul sukses: 02_pergerakan_nst.png
2026-07-20 22:08:07,894 - INFO - [run_pipeline] - Render modul sukses: 03_stacked_volume.png
2026-07-20 22:08:08,329 - INFO - [run_pipeline] - Render modul sukses: 04_histogram_kde.png
2026-07-20 22:08:09,333 - INFO - [run_pipeline] - Render modul sukses: 05_heatmap_korelasi.png
2026-07-20 22:08:09,371 - INFO - [run_pipeline] - Siklus pipeline berakhir dengan integritas penuh. Destinasi: G:\My Drive\skripsi\proje\proje\data\hasil_visualisasi


aditional, untuk perbaikan dan perbandingan 

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
================================================================================
FASE 3 - PENERAPAN BASELINE KOREKSI MANUAL DAN PERHITUNGAN NSt
Skripsi: Analisis Sentimen Media Sosial terhadap Volatilitas IHSG
         selama Demonstrasi Politik 2025 (Random Forest)
================================================================================

GAGASAN
    Dua masukan, satu keluaran, sekali jalan.

        MASUKAN 1  dataset_multilabel_final.csv  (data mentah berlabel model)
        MASUKAN 2  baseline_koreksi.csv          (tabel koreksi manual Anda)
        KELUARAN   dataset_final.csv + nst_harian.csv

    Koreksi manual diperlakukan sebagai anotasi peneliti yang menimpa label
    model. Skrip ini tidak menilai, tidak menebak, dan tidak punya kosakata
    apa pun. Ia hanya menerapkan keputusan yang sudah Anda ambil, lalu
    menghitung NSt harian dari label akhir.

DUA MODE, MASING-MASING SEKALI JALAN
    "TEMPLATE"  Tulis lembar kerja berisi row_key + teks + kolom kosong,
                untuk Anda labeli manual di Excel.
    "TERAPKAN"  Baca lembar kerja yang sudah terisi, terapkan ke dataset,
                hitung NSt harian.

FORMAT baseline_koreksi.csv YANG DIBACA
    Wajib   : row_key, label_koreksi
    Opsional: alasan, tingkat_keyakinan  (disimpan sebagai jejak audit)
    Kolom lain diabaikan, jadi lembar kerja hasil mode TEMPLATE bisa
    langsung dipakai apa adanya tanpa perlu dirapikan.

    Baris dengan label_koreksi kosong dianggap "tidak dikoreksi" dan
    dilewati. Anda tidak perlu menghapusnya dari lembar kerja.

CATATAN UNTUK BAB III
    Sebutkan berkas baseline_koreksi.csv sebagai instrumen anotasi manual,
    cantumkan jumlah baris yang ditinjau, jumlah yang dikoreksi, tanggal
    pembekuan, dan siapa pelabelnya. Bila hanya satu pelabel, nyatakan itu
    sebagai keterbatasan penelitian secara terbuka.

Dependensi: pandas
================================================================================
"""

from __future__ import annotations

import logging
import sys
from pathlib import Path
from typing import List

import pandas as pd

# ==================================================================
# KONFIGURASI
# ==================================================================
FOLDER = Path(r"G:\My Drive\skripsi\proje\proje\data\interim")

FILE_MENTAH = FOLDER / "dataset_multilabel_final.csv"
FILE_BASELINE = FOLDER / "baseline_koreksi.csv"
FILE_TEMPLATE = FOLDER / "baseline_koreksi_template.csv"
FILE_DATASET_FINAL = FOLDER / "dataset_final.csv"
FILE_NST = FOLDER / "nst_harian.csv"
FILE_LAPORAN = FOLDER / "laporan_fase3.txt"

MODE = "TEMPLATE"  # "TEMPLATE" | "TERAPKAN"

KEY_COL = "row_key"
TEXT_COL = "text_clean_bert"
DATE_COL = "tanggal_wib"

# Kolom label model yang dipakai sebagai dasar sebelum dikoreksi.
# Isi setelah Tahap E menentukan pemenang. Kosongkan hanya saat
# mode TEMPLATE, karena mode itu belum membutuhkannya.
KOLOM_DASAR = "label_hybrid5_mdhugol"

# --- Parameter mode TEMPLATE -------------------------------------
# "NEU_KERAS"  hanya baris yang neu di SELURUH kolom pelabel
# "NEU_DASAR"  baris yang neu pada KOLOM_DASAR saja
# "SEMUA"      seluruh baris, bila Anda ingin meninjau pos dan neg juga
CAKUPAN_TEMPLATE = "NEU_KERAS"

# --- Parameter mode TERAPKAN -------------------------------------
# True  : koreksi hanya sah pada baris yang label dasarnya "neu"
# False : koreksi boleh menimpa pos dan neg juga
HANYA_TIMPA_NEU = False

LABEL_SAH = {"pos", "neu", "neg"}

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
logger = logging.getLogger(__name__)


# ==================================================================
# PEMUATAN
# ==================================================================
def muat_mentah() -> pd.DataFrame:
    if not FILE_MENTAH.exists():
        raise FileNotFoundError(f"Data mentah tidak ditemukan: {FILE_MENTAH}")

    df = pd.read_csv(FILE_MENTAH, dtype={KEY_COL: str}, encoding="utf-8-sig")
    for k in (KEY_COL, TEXT_COL, DATE_COL):
        if k not in df.columns:
            raise KeyError(f"Kolom wajib '{k}' tidak ada pada data mentah.")
    if df[KEY_COL].duplicated().any():
        n = int(df[KEY_COL].duplicated().sum())
        raise ValueError(f"{n} row_key ganda pada data mentah. Perbaiki dahulu.")

    logger.info("Data mentah: %d baris, %d kolom", len(df), len(df.columns))
    return df


def kolom_label(df: pd.DataFrame) -> List[str]:
    return [c for c in df.columns if c.startswith("label_")]


# ==================================================================
# MODE: TEMPLATE
# ==================================================================
def mode_template() -> None:
    df = muat_mentah()
    labels = kolom_label(df)

    if CAKUPAN_TEMPLATE == "NEU_KERAS":
        if not labels:
            raise KeyError("Tidak ada kolom berawalan 'label_'.")
        mask = df[labels].eq("neu").all(axis=1)
        catatan = f"neu pada seluruh {len(labels)} kolom pelabel"
    elif CAKUPAN_TEMPLATE == "NEU_DASAR":
        if KOLOM_DASAR not in df.columns:
            raise KeyError(f"KOLOM_DASAR '{KOLOM_DASAR}' tidak ada.")
        mask = df[KOLOM_DASAR].eq("neu")
        catatan = f"neu pada {KOLOM_DASAR}"
    elif CAKUPAN_TEMPLATE == "SEMUA":
        mask = pd.Series(True, index=df.index)
        catatan = "seluruh baris"
    else:
        raise ValueError(f"CAKUPAN_TEMPLATE '{CAKUPAN_TEMPLATE}' tidak dikenal.")

    kerja = df.loc[mask, [KEY_COL, DATE_COL, TEXT_COL]].copy()
    kerja[TEXT_COL] = kerja[TEXT_COL].fillna("").astype(str)
    kerja = kerja[kerja[TEXT_COL].str.strip().ne("")]

    if KOLOM_DASAR in df.columns:
        kerja.insert(3, "label_model", df.loc[kerja.index, KOLOM_DASAR])

    # Kolom yang Anda isi manual. Sengaja dikosongkan.
    kerja["label_koreksi"] = ""
    kerja["alasan"] = ""
    kerja["tingkat_keyakinan"] = ""

    FOLDER.mkdir(parents=True, exist_ok=True)
    kerja.to_csv(FILE_TEMPLATE, index=False, encoding="utf-8-sig")

    logger.info("Cakupan template: %s", catatan)
    logger.info("Tersimpan: %s (%d baris)", FILE_TEMPLATE, len(kerja))
    logger.info(
        "Isi kolom label_koreksi dengan pos/neu/neg. Kosongkan bila tidak "
        "dikoreksi. Simpan sebagai %s, lalu set MODE = 'TERAPKAN'.",
        FILE_BASELINE.name,
    )


# ==================================================================
# MODE: TERAPKAN
# ==================================================================
def baca_baseline(kunci_sah: set) -> pd.DataFrame:
    if not FILE_BASELINE.exists():
        raise FileNotFoundError(
            f"Baseline koreksi tidak ditemukan: {FILE_BASELINE}. "
            "Jalankan MODE 'TEMPLATE' lebih dahulu."
        )

    bl = pd.read_csv(FILE_BASELINE, dtype={KEY_COL: str}, encoding="utf-8-sig")
    for k in (KEY_COL, "label_koreksi"):
        if k not in bl.columns:
            raise KeyError(f"Kolom wajib '{k}' tidak ada pada baseline koreksi.")

    bl["label_koreksi"] = (
        bl["label_koreksi"].fillna("").astype(str).str.strip().str.lower()
    )
    n_awal = len(bl)
    bl = bl[bl["label_koreksi"].ne("")]
    logger.info("Baris baseline terbaca: %d, terisi: %d", n_awal, len(bl))

    salah = sorted(set(bl["label_koreksi"]) - LABEL_SAH)
    if salah:
        raise ValueError(f"Nilai label_koreksi tidak sah: {salah}. Hanya pos/neu/neg.")
    if bl[KEY_COL].duplicated().any():
        ganda = bl.loc[bl[KEY_COL].duplicated(), KEY_COL].tolist()[:10]
        raise ValueError(f"row_key ganda pada baseline: {ganda}")

    yatim = sorted(set(bl[KEY_COL]) - kunci_sah)
    if yatim:
        logger.warning(
            "%d row_key pada baseline tidak ada di data mentah, diabaikan. Contoh: %s",
            len(yatim),
            yatim[:5],
        )
        bl = bl[~bl[KEY_COL].isin(yatim)]

    return bl


def mode_terapkan() -> None:
    df = muat_mentah()
    if KOLOM_DASAR not in df.columns:
        raise KeyError(
            f"KOLOM_DASAR '{KOLOM_DASAR}' tidak ada pada data mentah. "
            "Isi dengan kolom pemenang Tahap E."
        )

    bl = baca_baseline(set(df[KEY_COL]))
    peta = dict(zip(bl[KEY_COL], bl["label_koreksi"]))

    df["label_final"] = df[KOLOM_DASAR]
    usul = df[KEY_COL].map(peta)

    boleh = usul.notna()
    if HANYA_TIMPA_NEU:
        boleh &= df[KOLOM_DASAR].eq("neu")
    berubah = boleh & usul.ne(df[KOLOM_DASAR])

    df.loc[boleh, "label_final"] = usul[boleh]
    df["dikoreksi_manual"] = berubah

    # Jejak audit ringkas, tanpa teks bukti.
    for kol in ("alasan", "tingkat_keyakinan"):
        if kol in bl.columns:
            df[f"koreksi_{kol}"] = (
                df[KEY_COL].map(dict(zip(bl[KEY_COL], bl[kol]))).where(berubah, "")
            )

    FOLDER.mkdir(parents=True, exist_ok=True)
    df.to_csv(FILE_DATASET_FINAL, index=False, encoding="utf-8-sig")

    nst = hitung_nst(df)
    nst.to_csv(FILE_NST, index=False, encoding="utf-8-sig")

    lap = susun_laporan(df, bl, berubah, nst)
    FILE_LAPORAN.write_text(lap, encoding="utf-8")
    print("\n" + lap + "\n")

    logger.info("Tersimpan: %s (%d baris)", FILE_DATASET_FINAL, len(df))
    logger.info("Tersimpan: %s (%d hari)", FILE_NST, len(nst))


# ==================================================================
# PERHITUNGAN NSt
# ==================================================================
def hitung_nst(df: pd.DataFrame) -> pd.DataFrame:
    d = df.copy()
    d["tanggal"] = pd.to_datetime(d[DATE_COL], errors="coerce").dt.date
    d = d[d["tanggal"].notna()]

    tab = (
        d.pivot_table(
            index="tanggal",
            columns="label_final",
            values=KEY_COL,
            aggfunc="count",
            fill_value=0,
        )
        .reindex(columns=["pos", "neu", "neg"], fill_value=0)
        .reset_index()
    )
    tab["total"] = tab[["pos", "neu", "neg"]].sum(axis=1)
    # NSt = (pos - neg) / total, rentang -1 sampai +1 sesuai Bab III.
    tab["NSt"] = (tab["pos"] - tab["neg"]) / tab["total"].where(tab["total"] > 0)
    tab["NSt"] = tab["NSt"].fillna(0.0).round(6)
    return tab.sort_values("tanggal").reset_index(drop=True)


# ==================================================================
# LAPORAN
# ==================================================================
def susun_laporan(
    df: pd.DataFrame, bl: pd.DataFrame, berubah: pd.Series, nst: pd.DataFrame
) -> str:
    n = len(df)
    arah = (
        df.loc[berubah]
        .groupby([KOLOM_DASAR, "label_final"])
        .size()
        .sort_values(ascending=False)
    )
    b = [
        "LAPORAN FASE 3 - KOREKSI MANUAL DAN NSt HARIAN",
        "=" * 66,
        f"Data mentah          : {FILE_MENTAH.name} ({n} baris)",
        f"Baseline koreksi     : {FILE_BASELINE.name} ({len(bl)} entri terisi)",
        f"Kolom dasar          : {KOLOM_DASAR}",
        f"Hanya timpa neutral  : {HANYA_TIMPA_NEU}",
        f"Baris benar berubah  : {int(berubah.sum())} "
        f"({berubah.sum() / n * 100:.2f}% dari seluruh data)",
        "",
        "ARAH PERUBAHAN LABEL",
        "-" * 66,
    ]
    b += [f"  {a} -> {c} : {v}" for (a, c), v in arah.items()] or ["  (tidak ada)"]
    b += [
        "",
        "SEBARAN LABEL SEBELUM DAN SESUDAH",
        "-" * 66,
        f"  {'kelas':6s} {'sebelum':>9s} {'sesudah':>9s} {'selisih':>9s}",
    ]
    for k in ("pos", "neu", "neg"):
        s = int(df[KOLOM_DASAR].eq(k).sum())
        t = int(df["label_final"].eq(k).sum())
        b.append(f"  {k:6s} {s:9d} {t:9d} {t - s:+9d}")
    b += [
        "",
        "RINGKASAN NSt HARIAN",
        "-" * 66,
        f"  Jumlah hari    : {len(nst)}",
        f"  Rentang tanggal: {nst['tanggal'].min()} s.d. {nst['tanggal'].max()}",
        f"  NSt minimum    : {nst['NSt'].min():.4f}",
        f"  NSt maksimum   : {nst['NSt'].max():.4f}",
        f"  NSt rata-rata  : {nst['NSt'].mean():.4f}",
        "",
        "YANG WAJIB DILAPORKAN DI BAB III",
        "-" * 66,
        "1. Jumlah baris yang ditinjau manual dan jumlah yang dikoreksi.",
        "2. Kriteria yang Anda pakai untuk memutuskan sebuah teks ironis.",
        "3. Jumlah pelabel. Bila hanya satu orang, nyatakan sebagai",
        "   keterbatasan dan jangan mengklaim reliabilitas antar-pelabel.",
        "4. Tanggal pembekuan baseline_koreksi.csv, agar Bab IV terulangkan.",
    ]
    return "\n".join(b)


# ==================================================================
# ENTRY POINT
# ==================================================================
def main() -> int:
    peta = {"TEMPLATE": mode_template, "TERAPKAN": mode_terapkan}
    if MODE not in peta:
        logger.critical("MODE '%s' tidak dikenal. Pilih: %s", MODE, ", ".join(peta))
        return 1
    try:
        peta[MODE]()
    except (FileNotFoundError, KeyError, ValueError) as e:
        logger.critical("Gagal: %s", e)
        return 1
    return 0


if __name__ == "__main__":
    sys.exit(main())